In [1]:
from pathlib import Path

project_folder = Path(r"D:\ai-fmcg-sales-insights")

print("Is project file exist：", project_folder.exists())

if project_folder.exists():
    print("\ndata content of file：")
    for item in sorted((project_folder / "data").iterdir()):
        print(item.name)

Is project file exist： True

data content of file：
processed
raw


In [2]:
processed_folder = project_folder / "data" / "processed"

for item in sorted(processed_folder.rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"{item.relative_to(processed_folder)}  |  {size_mb:.2f} MB")

calendar_adjusted_test_metrics.csv  |  0.00 MB
calendar_adjusted_test_predictions.csv  |  0.08 MB
calendar_adjusted_validation_metrics.csv  |  0.00 MB
calendar_adjusted_validation_predictions.csv  |  0.09 MB
daily_store_dept_sales.csv  |  0.86 MB
dashboard_alerts_export.csv  |  0.01 MB
fmcg_sales_CA_1.csv.gz  |  8.94 MB
fmcg_sales_CA_2.csv.gz  |  8.49 MB
fmcg_sales_CA_3.csv.gz  |  9.20 MB
fmcg_sales_CA_4.csv.gz  |  8.49 MB
fmcg_sales_TX_1.csv.gz  |  8.64 MB
fmcg_sales_TX_2.csv.gz  |  8.79 MB
fmcg_sales_TX_3.csv.gz  |  8.76 MB
fmcg_sales_WI_1.csv.gz  |  8.89 MB
fmcg_sales_WI_2.csv.gz  |  8.99 MB
fmcg_sales_WI_3.csv.gz  |  8.68 MB
forecast_monthly_pattern_backtests.csv  |  0.00 MB
forecast_monthly_pattern_test_metrics.csv  |  0.00 MB
forecast_monthly_pattern_test_predictions.csv  |  0.14 MB
forecast_monthly_pattern_validation_metrics.csv  |  0.00 MB
forecast_rolling_backtest_results.csv  |  0.00 MB
future_sales_forecast_calendar_adjusted.csv  |  0.09 MB
future_sales_forecast_monthly_patt

In [3]:
import pandas as pd

sales_file = processed_folder / "daily_store_dept_sales.csv"
sales = pd.read_csv(sales_file)

print("Rows and columns:", sales.shape)

print("\nColumn names:")
print(sales.columns.tolist())

print("\nData types:")
print(sales.dtypes)

display(sales.head())

Rows and columns: (21900, 6)

Column names:
['SALES_DATE', 'STATE_ID', 'STORE_ID', 'DEPT_ID', 'TOTAL_UNITS_SOLD', 'TOTAL_SALES_REVENUE']

Data types:
SALES_DATE              object
STATE_ID                object
STORE_ID                object
DEPT_ID                 object
TOTAL_UNITS_SOLD         int64
TOTAL_SALES_REVENUE    float64
dtype: object


,SALES_DATE,STATE_ID,STORE_ID,DEPT_ID,TOTAL_UNITS_SOLD,TOTAL_SALES_REVENUE
0,2014-05-24,CA,CA_1,FOODS_1,343,750.20
1,2014-05-25,CA,CA_1,FOODS_1,379,683.36
2,2014-05-26,CA,CA_1,FOODS_1,250,586.27
3,2014-05-27,CA,CA_1,FOODS_1,207,442.35
4,2014-05-28,CA,CA_1,FOODS_1,197,502.52


In [4]:
sales["SALES_DATE"] = pd.to_datetime(
    sales["SALES_DATE"], errors="coerce"
)

print("Rows:", len(sales))
print("Start date:", sales["SALES_DATE"].min())
print("End date:", sales["SALES_DATE"].max())
print("Stores:", sales["STORE_ID"].nunique())
print("Departments:", sales["DEPT_ID"].nunique())

print("\nMissing values:")
print(sales.isna().sum())

key_columns = ["SALES_DATE", "STORE_ID", "DEPT_ID"]
print("\nDuplicate keys:", sales.duplicated(key_columns).sum())

value_columns = ["TOTAL_UNITS_SOLD", "TOTAL_SALES_REVENUE"]
print("\nNegative values:")
print((sales[value_columns] < 0).sum())

Rows: 21900
Start date: 2014-05-24 00:00:00
End date: 2016-05-22 00:00:00
Stores: 10
Departments: 3

Missing values:
SALES_DATE             0
STATE_ID               0
STORE_ID               0
DEPT_ID                0
TOTAL_UNITS_SOLD       0
TOTAL_SALES_REVENUE    0
dtype: int64

Duplicate keys: 0

Negative values:
TOTAL_UNITS_SOLD       0
TOTAL_SALES_REVENUE    0
dtype: int64


In [5]:
sales["SALES_DATE"] = pd.to_datetime(
    sales["SALES_DATE"], errors="raise"
).dt.normalize()

expected_dates = pd.date_range(
    sales["SALES_DATE"].min(),
    sales["SALES_DATE"].max(),
    freq="D"
)

expected_index = pd.MultiIndex.from_product(
    [
        expected_dates,
        sorted(sales["STORE_ID"].unique()),
        sorted(sales["DEPT_ID"].unique())
    ],
    names=["SALES_DATE", "STORE_ID", "DEPT_ID"]
)

actual_index = pd.MultiIndex.from_frame(
    sales[["SALES_DATE", "STORE_ID", "DEPT_ID"]]
)

missing_combinations = expected_index.difference(actual_index)

print("Date type:", sales["SALES_DATE"].dtype)
print("Expected days:", len(expected_dates))
print("Expected rows:", len(expected_index))
print("Actual rows:", len(sales))
print("Missing combinations:", len(missing_combinations))

if len(missing_combinations) > 0:
    display(missing_combinations.to_frame(index=False).head(10))

Date type: datetime64[ns]
Expected days: 730
Expected rows: 21900
Actual rows: 21900
Missing combinations: 0


In [6]:
report_end = pd.Timestamp("2016-05-22")
report_start = report_end - pd.Timedelta(days=6)

previous_end = report_start - pd.Timedelta(days=1)
previous_start = previous_end - pd.Timedelta(days=6)

current_week = sales.loc[
    sales["SALES_DATE"].between(report_start, report_end)
].copy()

previous_week = sales.loc[
    sales["SALES_DATE"].between(previous_start, previous_end)
].copy()

print(
    "Report period:",
    report_start.date(),
    "to",
    report_end.date()
)

print(
    "Comparison period:",
    previous_start.date(),
    "to",
    previous_end.date()
)

print("\nCurrent week rows:", len(current_week))
print("Previous week rows:", len(previous_week))

assert len(current_week) == 210, "Current week is incomplete."
assert len(previous_week) == 210, "Previous week is incomplete."

print("\nBoth periods contain 7 days x 10 stores x 3 departments.")

Report period: 2016-05-16 to 2016-05-22
Comparison period: 2016-05-09 to 2016-05-15

Current week rows: 210
Previous week rows: 210

Both periods contain 7 days x 10 stores x 3 departments.


In [7]:
def calculate_kpis(data):
    units = data["TOTAL_UNITS_SOLD"].sum()
    revenue = data["TOTAL_SALES_REVENUE"].sum()

    return {
        "Units sold": float(units),
        "Sales revenue (USD)": float(revenue),
        "Revenue per unit (USD)": (
            float(revenue / units) if units != 0 else None
        )
    }


current_kpis = calculate_kpis(current_week)
previous_kpis = calculate_kpis(previous_week)

kpi_summary = pd.DataFrame({
    "Previous week": previous_kpis,
    "Current week": current_kpis
})

kpi_summary["Change"] = (
    kpi_summary["Current week"] - kpi_summary["Previous week"]
)

kpi_summary["WoW change (%)"] = (
    kpi_summary["Change"]
    / kpi_summary["Previous week"].replace(0, float("nan"))
    * 100
)

display(kpi_summary.round(2))

,Previous week,Current week,Change,WoW change (%)
Units sold,225812.00,202535.00,-23277.00,-10.31
Sales revenue (USD),613002.71,539795.10,-73207.61,-11.94
Revenue per unit (USD),2.71,2.67,-0.05,-1.82


In [8]:
current_store = current_week.groupby("STORE_ID")[
    "TOTAL_SALES_REVENUE"
].sum()

previous_store = previous_week.groupby("STORE_ID")[
    "TOTAL_SALES_REVENUE"
].sum()

store_summary = pd.DataFrame({
    "Previous revenue": previous_store,
    "Current revenue": current_store
})

store_summary["Revenue change"] = (
    store_summary["Current revenue"]
    - store_summary["Previous revenue"]
)

store_summary["WoW change (%)"] = (
    store_summary["Revenue change"]
    / store_summary["Previous revenue"].replace(0, float("nan"))
    * 100
)

store_summary = store_summary.sort_values("Revenue change")

display(store_summary.round(2))

,Previous revenue,Current revenue,Revenue change,WoW change (%)
STORE_ID,,,,
WI_2,95506.05,63111.49,-32394.56,-33.92
WI_3,61222.07,48772.91,-12449.16,-20.33
WI_1,57511.38,50522.98,-6988.40,-12.15
CA_3,79228.48,72725.13,-6503.35,-8.21
TX_3,56265.79,51541.83,-4723.96,-8.40
TX_1,46182.45,41587.51,-4594.94,-9.95
TX_2,54522.58,50360.27,-4162.31,-7.63
CA_1,64439.94,61744.97,-2694.97,-4.18
CA_4,37324.84,36309.37,-1015.47,-2.72


In [9]:
current_dept = current_week.groupby("DEPT_ID")[
    "TOTAL_SALES_REVENUE"
].sum()

previous_dept = previous_week.groupby("DEPT_ID")[
    "TOTAL_SALES_REVENUE"
].sum()

department_summary = pd.DataFrame({
    "Previous revenue": previous_dept,
    "Current revenue": current_dept
})

department_summary["Revenue change"] = (
    department_summary["Current revenue"]
    - department_summary["Previous revenue"]
)

department_summary["WoW change (%)"] = (
    department_summary["Revenue change"]
    / department_summary["Previous revenue"].replace(0, float("nan"))
    * 100
)

department_summary = department_summary.sort_values("Revenue change")

display(department_summary.round(2))

,Previous revenue,Current revenue,Revenue change,WoW change (%)
DEPT_ID,,,,
FOODS_2,168175.53,130507.89,-37667.64,-22.40
FOODS_3,373318.11,340031.34,-33286.77,-8.92
FOODS_1,71509.07,69255.87,-2253.20,-3.15


In [10]:
import math

checks = {
    "Current revenue: stores vs total": math.isclose(
        store_summary["Current revenue"].sum(),
        current_kpis["Sales revenue (USD)"],
        abs_tol=0.01,
        rel_tol=0
    ),
    "Current revenue: departments vs total": math.isclose(
        department_summary["Current revenue"].sum(),
        current_kpis["Sales revenue (USD)"],
        abs_tol=0.01,
        rel_tol=0
    ),
    "Previous revenue: stores vs total": math.isclose(
        store_summary["Previous revenue"].sum(),
        previous_kpis["Sales revenue (USD)"],
        abs_tol=0.01,
        rel_tol=0
    ),
    "Previous revenue: departments vs total": math.isclose(
        department_summary["Previous revenue"].sum(),
        previous_kpis["Sales revenue (USD)"],
        abs_tol=0.01,
        rel_tol=0
    )
}

for check_name, passed in checks.items():
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

assert all(checks.values()), "Revenue reconciliation failed."

Current revenue: stores vs total: PASS
Current revenue: departments vs total: PASS
Previous revenue: stores vs total: PASS
Previous revenue: departments vs total: PASS


In [11]:
import json

evidence_package = {
    "schema_version": "1.0",
    "report_id": f"weekly_sales_{report_end:%Y%m%d}",
    "report_type": "Historical weekly sales review",
    "report_period": {
        "start": str(report_start.date()),
        "end": str(report_end.date())
    },
    "comparison_period": {
        "start": str(previous_start.date()),
        "end": str(previous_end.date())
    },
    "source": {
        "file": sales_file.relative_to(project_folder).as_posix(),
        "grain": "One row per date, store and department",
        "currency": "USD"
    },
    "overall_metrics": [
        {
            "evidence_id": f"overall_{index}",
            "metric": metric,
            "previous_value": float(row["Previous week"]),
            "current_value": float(row["Current week"]),
            "change": float(row["Change"]),
            "wow_change_pct": (
                None if pd.isna(row["WoW change (%)"])
                else float(row["WoW change (%)"])
            )
        }
        for index, (metric, row) in enumerate(
            kpi_summary.iterrows(), start=1
        )
    ],
    "store_metrics": [
        {
            "evidence_id": f"store_{store_id}",
            "store_id": store_id,
            **{key: float(value) for key, value in row.items()}
        }
        for store_id, row in store_summary.iterrows()
    ],
    "department_metrics": [
        {
            "evidence_id": f"department_{dept_id}",
            "department_id": dept_id,
            **{key: float(value) for key, value in row.items()}
        }
        for dept_id, row in department_summary.iterrows()
    ],
    "reconciliation_checks": checks,
    "limitations": [
        "Historical dataset; this is not a live business report.",
        "Week-over-week changes do not establish causes.",
        "Revenue per unit reflects both prices and product mix.",
        "Week-over-week declines are not model-based alerts.",
        "Forecasts and model-based alerts are not included yet.",
        "Human review is required before approval."
    ]
}

evidence_json = json.dumps(
    evidence_package,
    indent=2,
    allow_nan=False
)

print(evidence_json)

{
  "schema_version": "1.0",
  "report_id": "weekly_sales_20160522",
  "report_type": "Historical weekly sales review",
  "report_period": {
    "start": "2016-05-16",
    "end": "2016-05-22"
  },
  "comparison_period": {
    "start": "2016-05-09",
    "end": "2016-05-15"
  },
  "source": {
    "file": "data/processed/daily_store_dept_sales.csv",
    "grain": "One row per date, store and department",
    "currency": "USD"
  },
  "overall_metrics": [
    {
      "evidence_id": "overall_1",
      "metric": "Units sold",
      "previous_value": 225812.0,
      "current_value": 202535.0,
      "change": -23277.0,
      "wow_change_pct": -10.308132428746037
    },
    {
      "evidence_id": "overall_2",
      "metric": "Sales revenue (USD)",
      "previous_value": 613002.7100000001,
      "current_value": 539795.1,
      "change": -73207.6100000001,
      "wow_change_pct": -11.942461070033458
    },
    {
      "evidence_id": "overall_3",
      "metric": "Revenue per unit (USD)",
      "pr

In [12]:
evidence_folder = project_folder / "reports" / "evidence"
evidence_folder.mkdir(parents=True, exist_ok=True)

evidence_file = (
    evidence_folder
    / f"{evidence_package['report_id']}_evidence.json"
)

evidence_file.write_text(evidence_json, encoding="utf-8")

saved_evidence = json.loads(
    evidence_file.read_text(encoding="utf-8")
)

assert saved_evidence == evidence_package, "Saved evidence does not match."

print("Evidence saved successfully.")
print("File:", evidence_file)
print("Report ID:", saved_evidence["report_id"])
print(
    "Evidence records:",
    sum(
        len(saved_evidence[key])
        for key in [
            "overall_metrics",
            "store_metrics",
            "department_metrics"
        ]
    )
)

Evidence saved successfully.
File: D:\ai-fmcg-sales-insights\reports\evidence\weekly_sales_20160522_evidence.json
Report ID: weekly_sales_20160522
Evidence records: 16


In [13]:
report_instructions = """
You are preparing a historical weekly sales review for a sales manager.

Use only the supplied evidence package.

Rules:
1. Write in clear, concise business English.
2. Use the report and comparison periods exactly as supplied.
3. Support every factual statement with relevant evidence_ids.
4. Copy numerical facts from the evidence. Do not invent figures.
5. Distinguish units sold, revenue in USD, and revenue per unit.
6. Describe week-over-week changes without claiming their causes.
7. Do not describe week-over-week declines as model-based alerts.
8. Do not infer stockouts, promotions, profitability, or price changes.
9. Present follow-up questions as investigations, not confirmed findings.
10. Do not combine store and department totals as separate contributions
    to the same overall change; they describe overlapping dimensions.
11. State that the report uses historical data and requires human review.
12. Return valid JSON only, without Markdown fences.

Required output structure:
{
  "report_id": "Copy the report_id from the evidence",
  "status": "draft",
  "executive_summary": [
    {
      "text": "A concise factual statement",
      "evidence_ids": ["A relevant evidence_id"]
    }
  ],
  "key_findings": [
    {
      "text": "A store or department finding",
      "evidence_ids": ["A relevant evidence_id"]
    }
  ],
  "follow_up_questions": [
    {
      "question": "A question for further investigation",
      "evidence_ids": ["The evidence motivating this question"]
    }
  ],
  "limitations": [
    "A relevant limitation"
  ]
}

Include:
- Two or three executive summary statements.
- Up to four key findings.
- Two follow-up questions.
"""

print(report_instructions)


You are preparing a historical weekly sales review for a sales manager.

Use only the supplied evidence package.

Rules:
1. Write in clear, concise business English.
2. Use the report and comparison periods exactly as supplied.
3. Support every factual statement with relevant evidence_ids.
4. Copy numerical facts from the evidence. Do not invent figures.
5. Distinguish units sold, revenue in USD, and revenue per unit.
6. Describe week-over-week changes without claiming their causes.
7. Do not describe week-over-week declines as model-based alerts.
8. Do not infer stockouts, promotions, profitability, or price changes.
9. Present follow-up questions as investigations, not confirmed findings.
10. Do not combine store and department totals as separate contributions
    to the same overall change; they describe overlapping dimensions.
11. State that the report uses historical data and requires human review.
12. Return valid JSON only, without Markdown fences.

Required output structure:
{

In [14]:
prompt_folder = project_folder / "prompts"
prompt_folder.mkdir(parents=True, exist_ok=True)

prompt_file = prompt_folder / "weekly_sales_review_v1.txt"
prompt_file.write_text(
    report_instructions.strip(),
    encoding="utf-8"
)

print("Prompt saved successfully.")
print("File:", prompt_file)

Prompt saved successfully.
File: D:\ai-fmcg-sales-insights\prompts\weekly_sales_review_v1.txt


In [15]:
%pip install openai

In [16]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API key loaded for this session.")

Enter your OpenAI API key: ········
API key loaded for this session.


In [17]:
%pip install --upgrade typing_extensions

Note: you may need to restart the kernel to use updated packages.


In [18]:
from typing_extensions import Sentinel
from openai import OpenAI

print("Imports successful.")

Imports successful.


In [19]:
import os
from getpass import getpass
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(timeout=30.0, max_retries=0)

print("Client ready.")

Enter your OpenAI API key: ········
Client ready.


In [20]:
test_response = client.responses.create(
    model="gpt-4.1-mini",
    input="Reply with exactly: API connection successful.",
    max_output_tokens=50,
    store=False
)

print(test_response.output_text)


API connection successful.


In [21]:
import json
from pathlib import Path

project_folder = Path(r"D:\ai-fmcg-sales-insights")

evidence_file = (
    project_folder
    / "reports"
    / "evidence"
    / "weekly_sales_20160522_evidence.json"
)

prompt_file = (
    project_folder
    / "prompts"
    / "weekly_sales_review_v1.txt"
)

evidence_package = json.loads(
    evidence_file.read_text(encoding="utf-8")
)

report_instructions = prompt_file.read_text(encoding="utf-8")

print("Report ID:", evidence_package["report_id"])
print("Prompt loaded:", bool(report_instructions.strip()))
print(
    "Reconciliation passed:",
    all(evidence_package["reconciliation_checks"].values())
)

Report ID: weekly_sales_20160522
Prompt loaded: True
Reconciliation passed: True


In [23]:
from datetime import datetime, timezone

report_response = client.responses.create(
    model="gpt-4.1-mini",
    instructions=report_instructions,
    input=(
    "Generate the weekly sales review as a JSON object "
    "using only the following evidence package:\n\n"
    + json.dumps(evidence_package, allow_nan=False)
),
    text={"format": {"type": "json_object"}},
    max_output_tokens=2000,
    store=False
)

draft_folder = project_folder / "reports" / "drafts"
draft_folder.mkdir(parents=True, exist_ok=True)

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%S%fZ")
draft_file = draft_folder / (
    f"{evidence_package['report_id']}_{run_id}_draft.json"
)

draft_file.write_text(
    report_response.output_text,
    encoding="utf-8"
)

print("Response status:", report_response.status)
print("Draft saved:", draft_file)
print("\nGenerated draft:")
print(report_response.output_text)

Response status: completed
Draft saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T0330056875Z_draft.json

Generated draft:
{
  "report_id": "weekly_sales_20160522",
  "status": "draft",
  "executive_summary": [
    {
      "text": "Total units sold decreased by 23,277 units, a 10.31% decline week-over-week from 225,812 to 202,535 units.",
      "evidence_ids": ["overall_1"]
    },
    {
      "text": "Total sales revenue fell by $73,207.61, representing an 11.94% decline from $613,002.71 to $539,795.10.",
      "evidence_ids": ["overall_2"]
    },
    {
      "text": "Revenue per unit decreased slightly by approximately $0.05 or 1.82%, from $2.71 to $2.67.",
      "evidence_ids": ["overall_3"]
    }
  ],
  "key_findings": [
    {
      "text": "Store WI_2 experienced the largest revenue decline of $32,394.56, a 33.92% decrease week-over-week.",
      "evidence_ids": ["store_WI_2"]
    },
    {
      "text": "Store CA_2 showed a revenue increase of $2,319.5

In [24]:
draft = json.loads(draft_file.read_text(encoding="utf-8"))

valid_evidence_ids = {
    item["evidence_id"]
    for section in [
        "overall_metrics",
        "store_metrics",
        "department_metrics"
    ]
    for item in evidence_package[section]
}

validation_errors = []

if draft.get("report_id") != evidence_package["report_id"]:
    validation_errors.append("Report ID mismatch.")

if draft.get("status") != "draft":
    validation_errors.append("Report status must be draft.")

for section, text_field in [
    ("executive_summary", "text"),
    ("key_findings", "text"),
    ("follow_up_questions", "question")
]:
    items = draft.get(section)

    if not isinstance(items, list) or not items:
        validation_errors.append(f"{section}: missing or invalid list.")
        continue

    for index, item in enumerate(items, start=1):
        label = f"{section}[{index}]"

        if not isinstance(item, dict):
            validation_errors.append(f"{label}: invalid item.")
            continue

        text = item.get(text_field)
        if not isinstance(text, str) or not text.strip():
            validation_errors.append(f"{label}: missing text.")

        references = item.get("evidence_ids")

        if not isinstance(references, list) or not references:
            validation_errors.append(f"{label}: missing evidence IDs.")
            continue

        for reference in references:
            if not isinstance(reference, str):
                validation_errors.append(f"{label}: invalid evidence ID.")
            elif reference not in valid_evidence_ids:
                validation_errors.append(
                    f"{label}: unknown evidence ID '{reference}'."
                )

if validation_errors:
    print("Basic validation: FAIL")
    for error in validation_errors:
        print("-", error)
else:
    print("Basic validation: PASS")
    print("Report ID, draft status and evidence references checked.")

print("\nNumerical and semantic validation: NOT YET COMPLETED")
print("Human approval: PENDING")

Basic validation: PASS
Report ID, draft status and evidence references checked.

Numerical and semantic validation: NOT YET COMPLETED
Human approval: PENDING


In [25]:
evidence_lookup = {
    item["evidence_id"]: item
    for section in [
        "overall_metrics",
        "store_metrics",
        "department_metrics"
    ]
    for item in evidence_package[section]
}

for section, text_field in [
    ("executive_summary", "text"),
    ("key_findings", "text"),
    ("follow_up_questions", "question")
]:
    print(f"\n{'=' * 60}")
    print(section.upper())

    for index, item in enumerate(draft[section], start=1):
        print(f"\nStatement {index}:")
        print(item[text_field])

        print("\nReferenced evidence:")
        for evidence_id in item["evidence_ids"]:
            print(json.dumps(
                evidence_lookup[evidence_id],
                indent=2,
                allow_nan=False
            ))


EXECUTIVE_SUMMARY

Statement 1:
Total units sold decreased by 23,277 units, a 10.31% decline week-over-week from 225,812 to 202,535 units.

Referenced evidence:
{
  "evidence_id": "overall_1",
  "metric": "Units sold",
  "previous_value": 225812.0,
  "current_value": 202535.0,
  "change": -23277.0,
  "wow_change_pct": -10.308132428746037
}

Statement 2:
Total sales revenue fell by $73,207.61, representing an 11.94% decline from $613,002.71 to $539,795.10.

Referenced evidence:
{
  "evidence_id": "overall_2",
  "metric": "Sales revenue (USD)",
  "previous_value": 613002.7100000001,
  "current_value": 539795.1,
  "change": -73207.6100000001,
  "wow_change_pct": -11.942461070033458
}

Statement 3:
Revenue per unit decreased slightly by approximately $0.05 or 1.82%, from $2.71 to $2.67.

Referenced evidence:
{
  "evidence_id": "overall_3",
  "metric": "Revenue per unit (USD)",
  "previous_value": 2.7146595840787917,
  "current_value": 2.665194163971659,
  "change": -0.04946542010713273,
 

In [26]:
import re
import math
import pandas as pd

number_pattern = r"(?<![\w])[-+]?\d+(?:,\d{3})*(?:\.\d+)?%?"

numeric_results = []

for section, text_field in [
    ("executive_summary", "text"),
    ("key_findings", "text"),
    ("follow_up_questions", "question")
]:
    for index, item in enumerate(draft[section], start=1):
        statement = item[text_field]

        for evidence_id in item["evidence_ids"]:
            evidence = evidence_lookup[evidence_id]

            for id_field in ["store_id", "department_id"]:
                identifier = evidence.get(id_field)
                if identifier:
                    statement = statement.replace(identifier, "")

        for token in re.findall(number_pattern, statement):
            is_percentage = token.endswith("%")
            number_text = token.rstrip("%").replace(",", "")
            reported_value = abs(float(number_text))

            decimal_places = (
                len(number_text.split(".")[1])
                if "." in number_text else 0
            )

            candidates = []

            for evidence_id in item["evidence_ids"]:
                evidence = evidence_lookup[evidence_id]

                for field, value in evidence.items():
                    if type(value) not in (int, float):
                        continue

                    percentage_field = field in [
                        "wow_change_pct",
                        "WoW change (%)"
                    ]

                    if percentage_field == is_percentage:
                        candidates.append(abs(value))

            matched = any(
                math.isclose(
                    reported_value,
                    round(value, decimal_places),
                    rel_tol=0,
                    abs_tol=0.000001
                )
                for value in candidates
            )

            numeric_results.append({
                "section": section,
                "statement": index,
                "reported_number": token,
                "result": "MATCH" if matched else "REVIEW"
            })

numeric_check_table = pd.DataFrame(numeric_results)

display(numeric_check_table)

print(
    "Numbers requiring review:",
    (numeric_check_table["result"] == "REVIEW").sum()
)
print("Human approval: PENDING")

,section,statement,reported_number,result
0,executive_summary,1,"23,277",MATCH
1,executive_summary,1,10.31%,MATCH
2,executive_summary,1,"225,812",MATCH
3,executive_summary,1,"202,535",MATCH
4,executive_summary,2,"73,207.61",MATCH
5,executive_summary,2,11.94%,MATCH
6,executive_summary,2,"613,002.71",MATCH
7,executive_summary,2,"539,795.10",MATCH
8,executive_summary,3,0.05,MATCH
9,executive_summary,3,1.82%,MATCH


Numbers requiring review: 0
Human approval: PENDING


In [27]:
def find_unmatched_numbers(text, evidence):
    unmatched = []

    for token in re.findall(number_pattern, text):
        is_percentage = token.endswith("%")
        number_text = token.rstrip("%").replace(",", "")
        decimal_places = (
            len(number_text.split(".")[1])
            if "." in number_text else 0
        )

        candidates = [
            abs(value)
            for field, value in evidence.items()
            if type(value) in (int, float)
            and (
                (field == "wow_change_pct")
                == is_percentage
            )
        ]

        matched = any(
            math.isclose(
                abs(float(number_text)),
                round(value, decimal_places),
                rel_tol=0,
                abs_tol=0.000001
            )
            for value in candidates
        )

        if not matched:
            unmatched.append(token)

    return unmatched


original_text = draft["executive_summary"][0]["text"]
assert "202,535" in original_text, "Expected test value not found."

incorrect_text = original_text.replace("202,535", "999,999")
test_evidence = evidence_lookup["overall_1"]

original_errors = find_unmatched_numbers(original_text, test_evidence)
injected_errors = find_unmatched_numbers(incorrect_text, test_evidence)

assert original_errors == [], "Original statement unexpectedly failed."
assert "999,999" in injected_errors, "Injected error was not detected."

print("Original statement: PASS")
print("Injected incorrect number: DETECTED")
print("Flagged numbers:", injected_errors)
print("Original draft unchanged.")

Original statement: PASS
Injected incorrect number: DETECTED
Flagged numbers: ['999,999']
Original draft unchanged.


In [28]:
validation_record = {
    "report_id": evidence_package["report_id"],
    "draft_file": draft_file.relative_to(project_folder).as_posix(),
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    "basic_validation_errors": validation_errors,
    "numeric_screening": {
        "numbers_checked": len(numeric_check_table),
        "numbers_requiring_review": int(
            (numeric_check_table["result"] == "REVIEW").sum()
        ),
        "scope": "Rounded numeric matching against cited evidence only"
    },
    "error_injection_test": {
        "original_statement_passed": original_errors == [],
        "incorrect_number_detected": "999,999" in injected_errors
    },
    "manual_review_items": [
        "Add explicit report and comparison dates.",
        "Label monetary values as USD.",
        "Review evidence coverage for cross-store comparisons.",
        "Check direction, units, period attribution and ranking claims."
    ],
    "semantic_validation": "pending",
    "human_approval": "pending"
}

validation_folder = project_folder / "reports" / "validation"
validation_folder.mkdir(parents=True, exist_ok=True)

validation_file = (
    validation_folder / f"{draft_file.stem}_validation.json"
)

validation_file.write_text(
    json.dumps(validation_record, indent=2, allow_nan=False),
    encoding="utf-8"
)

print("Validation record saved:", validation_file)
print("Semantic validation: PENDING")
print("Human approval: PENDING")

Validation record saved: D:\ai-fmcg-sales-insights\reports\validation\weekly_sales_20160522_20261021T0330056875Z_draft_validation.json
Semantic validation: PENDING
Human approval: PENDING


In [29]:
from copy import deepcopy

revised_draft = deepcopy(draft)

revised_draft["report_period"] = deepcopy(
    evidence_package["report_period"]
)
revised_draft["comparison_period"] = deepcopy(
    evidence_package["comparison_period"]
)
revised_draft["currency"] = evidence_package["source"]["currency"]

for section, text_field in [
    ("executive_summary", "text"),
    ("key_findings", "text"),
    ("follow_up_questions", "question")
]:
    for item in revised_draft[section]:
        item[text_field] = item[text_field].replace("$", "USD ")

revised_draft["follow_up_questions"][1] = {
    "question": (
        "What should be investigated to understand "
        "the week-over-week revenue increase in store CA_2?"
    ),
    "evidence_ids": ["store_CA_2"]
}

revised_draft["revision"] = {
    "source_draft": draft_file.relative_to(project_folder).as_posix(),
    "method": "Programmatic edits specified during review",
    "changes": [
        "Added report and comparison periods from the evidence.",
        "Explicitly labelled currency as USD.",
        "Removed the cross-store comparison from the CA_2 question."
    ]
}

revised_draft["status"] = "draft"

revised_file = draft_file.with_name(
    f"{draft_file.stem}_revised.json"
)

revised_file.write_text(
    json.dumps(revised_draft, indent=2, allow_nan=False),
    encoding="utf-8"
)

print("Revised draft saved:", revised_file)
print("\nRevised report:")
print(json.dumps(revised_draft, indent=2, allow_nan=False))
print("\nHuman approval: PENDING")

Revised draft saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T0330056875Z_draft_revised.json

Revised report:
{
  "report_id": "weekly_sales_20160522",
  "status": "draft",
  "executive_summary": [
    {
      "text": "Total units sold decreased by 23,277 units, a 10.31% decline week-over-week from 225,812 to 202,535 units.",
      "evidence_ids": [
        "overall_1"
      ]
    },
    {
      "text": "Total sales revenue fell by USD 73,207.61, representing an 11.94% decline from USD 613,002.71 to USD 539,795.10.",
      "evidence_ids": [
        "overall_2"
      ]
    },
    {
      "text": "Revenue per unit decreased slightly by approximately USD 0.05 or 1.82%, from USD 2.71 to USD 2.67.",
      "evidence_ids": [
        "overall_3"
      ]
    }
  ],
  "key_findings": [
    {
      "text": "Store WI_2 experienced the largest revenue decline of USD 32,394.56, a 33.92% decrease week-over-week.",
      "evidence_ids": [
        "store_WI_2"
      ]
   

In [30]:
store_evidence = evidence_package["store_metrics"]

largest_decline = min(
    store_evidence,
    key=lambda item: item["Revenue change"]
)

assert largest_decline["store_id"] == "WI_2", (
    "WI_2 is not the store with the largest revenue decline."
)

ranking_finding = revised_draft["key_findings"][0]

assert ranking_finding["evidence_ids"] == ["store_WI_2"], (
    "Review the ranking finding before updating references."
)

ranking_finding["evidence_ids"] = [
    item["evidence_id"] for item in store_evidence
]

change_note = (
    "Verified the largest revenue decline against all stores "
    "and added the comparison evidence."
)

if change_note not in revised_draft["revision"]["changes"]:
    revised_draft["revision"]["changes"].append(change_note)

revised_file.write_text(
    json.dumps(revised_draft, indent=2, allow_nan=False),
    encoding="utf-8"
)

print("Largest revenue decline verified:", largest_decline["store_id"])
print("Comparison evidence records:", len(ranking_finding["evidence_ids"]))
print("Revised draft updated.")
print("Human approval: PENDING")

Largest revenue decline verified: WI_2
Comparison evidence records: 10
Revised draft updated.
Human approval: PENDING


# Weekly Sales Review

**Status: DRAFT — Pending human approval**

Report ID: weekly_sales_20160522

Report period: 2016-05-16 to 2016-05-22

Comparison period: 2016-05-09 to 2016-05-15

Currency: USD

## Executive Summary

- Total units sold decreased by 23,277 units, a 10.31% decline week-over-week from 225,812 to 202,535 units.
  Evidence: `overall_1`

- Total sales revenue fell by USD 73,207.61, representing an 11.94% decline from USD 613,002.71 to USD 539,795.10.
  Evidence: `overall_2`

- Revenue per unit decreased slightly by approximately USD 0.05 or 1.82%, from USD 2.71 to USD 2.67.
  Evidence: `overall_3`

## Key Findings

- Store WI_2 experienced the largest revenue decline of USD 32,394.56, a 33.92% decrease week-over-week.
  Evidence: `store_WI_2, store_WI_3, store_WI_1, store_CA_3, store_TX_3, store_TX_1, store_TX_2, store_CA_1, store_CA_4, store_CA_2`

- Store CA_2 showed a revenue increase of USD 2,319.51, a 3.82% week-over-week growth.
  Evidence: `store_CA_2`

- Department FOODS_2 had a revenue decline of USD 37,667.64, a 22.40% decrease compared with the previous week.
  Evidence: `department_FOODS_2`

- Department FOODS_3's revenue decreased by USD 33,286.77, an 8.92% decline week-over-week.
  Evidence: `department_FOODS_3`

## Follow-up Questions

- What factors contributed to the significant 33.92% revenue decrease in store WI_2 during the report period?
  Evidence: `store_WI_2`

- What should be investigated to understand the week-over-week revenue increase in store CA_2?
  Evidence: `store_CA_2`

## Limitations

- Historical dataset; this is not a live business report.
- Week-over-week changes do not establish causes.
- Revenue per unit reflects both prices and product mix.
- Week-over-week declines are not model-based alerts.
- Forecasts and model-based alerts are not included yet.
- Human review is required before approval.

Draft preview saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T0330056875Z_draft_revised.md


In [32]:
unit_revenue = evidence_lookup["overall_3"]

revised_draft["executive_summary"][2]["text"] = (
    f"Revenue per unit decreased by "
    f"USD {abs(unit_revenue['change']):.4f} "
    f"({abs(unit_revenue['wow_change_pct']):.2f}%), "
    f"from USD {unit_revenue['previous_value']:.4f} "
    f"to USD {unit_revenue['current_value']:.4f}."
)

revision_note = (
    "Displayed revenue per unit to four decimal places "
    "to reduce rounding ambiguity."
)

if revision_note not in revised_draft["revision"]["changes"]:
    revised_draft["revision"]["changes"].append(revision_note)

revised_file.write_text(
    json.dumps(revised_draft, indent=2, allow_nan=False),
    encoding="utf-8"
)

print(revised_draft["executive_summary"][2]["text"])
print("\nHuman approval: PENDING")

Revenue per unit decreased by USD 0.0495 (1.82%), from USD 2.7147 to USD 2.6652.

Human approval: PENDING


In [37]:
from IPython.display import Markdown, display

report = json.loads(revised_file.read_text(encoding="utf-8"))


def display_label(identifier):
    if identifier.startswith("FOODS_"):
        return identifier.replace("FOODS_", "Foods ")
    return identifier.replace("_", " ")


units = evidence_lookup["overall_1"]
revenue = evidence_lookup["overall_2"]
wi2 = evidence_lookup["store_WI_2"]
ca2 = evidence_lookup["store_CA_2"]
foods2 = evidence_lookup["department_FOODS_2"]

action_items = [
    {
        "action_id": "action_1",
        "action": (
            "Break down WI 2 revenue by department and date, "
            "comparing both weeks."
        ),
        "expected_output": (
            "Identify the departments and dates contributing "
            "most to the store's revenue decline."
        ),
        "evidence_ids": ["store_WI_2"]
    },
    {
        "action_id": "action_2",
        "action": (
            "Compare CA 2 department revenue across both weeks."
        ),
        "expected_output": (
            "Identify which departments contributed "
            "to the store's revenue growth."
        ),
        "evidence_ids": ["store_CA_2"]
    }
]

manager_preview = f"""
# Weekly Sales Review

**DRAFT | {report['report_period']['start']} to {report['report_period']['end']}**

Compared with {report['comparison_period']['start']} to {report['comparison_period']['end']}.
All monetary values are in USD.

## Performance

| Metric | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Revenue | {revenue['previous_value']:,.2f} | {revenue['current_value']:,.2f} | {revenue['change']:+,.2f} | {revenue['wow_change_pct']:+.2f}% |
| Units sold | {units['previous_value']:,.0f} | {units['current_value']:,.0f} | {units['change']:+,.0f} | {units['wow_change_pct']:+.2f}% |

## Key Findings

- **{display_label(wi2['store_id'])}** recorded the largest store revenue
  decline by amount: **{abs(wi2['Revenue change']):,.2f}**
  ({wi2['WoW change (%)']:.2f}%).
- **{display_label(foods2['department_id'])}** revenue fell
  **{abs(foods2['Revenue change']):,.2f}**
  ({foods2['WoW change (%)']:.2f}%).
- **{display_label(ca2['store_id'])}** revenue increased
  **{ca2['Revenue change']:,.2f}**
  (+{ca2['WoW change (%)']:.2f}%).

## Proposed Actions

| Action | Expected output |
| --- | --- |
"""

for item in action_items:
    manager_preview += (
        f"| {item['action']} | {item['expected_output']} |\n"
    )

manager_preview += """
\n*Historical data; causes are unconfirmed.
Actions are proposed and have not been completed.
Human approval is pending.*
"""

manager_preview_file = revised_file.with_name(
    f"{revised_file.stem}_manager_preview.md"
)

manager_preview_file.write_text(
    manager_preview.strip(),
    encoding="utf-8"
)

action_file = revised_file.with_name(
    f"{revised_file.stem}_proposed_actions.json"
)

action_file.write_text(
    json.dumps(
        {
            "report_id": report["report_id"],
            "status": "proposed",
            "source": "Review-stage actions; not LLM-generated",
            "actions": action_items
        },
        indent=2,
        allow_nan=False
    ),
    encoding="utf-8"
)

display(Markdown(manager_preview))
print("Manager preview saved:", manager_preview_file)
print("Proposed actions saved:", action_file)


# Weekly Sales Review

**DRAFT | 2016-05-16 to 2016-05-22**

Compared with 2016-05-09 to 2016-05-15.
All monetary values are in USD.

## Performance

| Metric | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Revenue | 613,002.71 | 539,795.10 | -73,207.61 | -11.94% |
| Units sold | 225,812 | 202,535 | -23,277 | -10.31% |

## Key Findings

- **WI 2** recorded the largest store revenue
  decline by amount: **32,394.56**
  (-33.92%).
- **Foods 2** revenue fell
  **37,667.64**
  (-22.40%).
- **CA 2** revenue increased
  **2,319.51**
  (+3.82%).

## Proposed Actions

| Action | Expected output |
| --- | --- |
| Break down WI 2 revenue by department and date, comparing both weeks. | Identify the departments and dates contributing most to the store's revenue decline. |
| Compare CA 2 department revenue across both weeks. | Identify which departments contributed to the store's revenue growth. |


*Historical data; causes are unconfirmed.
Actions are proposed and have not been completed.
Human approval is pending.*


Manager preview saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T0330056875Z_draft_revised_manager_preview.md
Proposed actions saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T0330056875Z_draft_revised_proposed_actions.json


In [38]:
import pandas as pd

department_performance = pd.DataFrame(
    evidence_package["department_metrics"]
).rename(columns={
    "department_id": "Department",
    "Previous revenue": "Previous week",
    "Current revenue": "Current week",
    "Revenue change": "Change",
    "WoW change (%)": "Change (%)"
})

department_performance["Department"] = (
    department_performance["Department"].map(display_label)
)

department_performance = department_performance[
    [
        "Department",
        "Previous week",
        "Current week",
        "Change",
        "Change (%)"
    ]
].sort_values("Department")

print("Department performance — Revenue in USD")
display(department_performance.round(2))


sales["SALES_DATE"] = pd.to_datetime(
    sales["SALES_DATE"], errors="raise"
).dt.normalize()

report_start = pd.Timestamp(evidence_package["report_period"]["start"])
report_end = pd.Timestamp(evidence_package["report_period"]["end"])
previous_start = pd.Timestamp(
    evidence_package["comparison_period"]["start"]
)
previous_end = pd.Timestamp(
    evidence_package["comparison_period"]["end"]
)

current_store_dept = sales.loc[
    sales["SALES_DATE"].between(report_start, report_end)
].groupby(["STORE_ID", "DEPT_ID"])["TOTAL_SALES_REVENUE"].sum()

previous_store_dept = sales.loc[
    sales["SALES_DATE"].between(previous_start, previous_end)
].groupby(["STORE_ID", "DEPT_ID"])["TOTAL_SALES_REVENUE"].sum()

store_department_performance = pd.DataFrame({
    "Previous week": previous_store_dept,
    "Current week": current_store_dept
})

store_department_performance["Change"] = (
    store_department_performance["Current week"]
    - store_department_performance["Previous week"]
)

store_department_performance["Change (%)"] = (
    store_department_performance["Change"]
    / store_department_performance["Previous week"].replace(
        0, float("nan")
    )
    * 100
)

focus_stores = [
    min(
        evidence_package["store_metrics"],
        key=lambda item: item["Revenue change"]
    )["store_id"],
    max(
        evidence_package["store_metrics"],
        key=lambda item: item["Revenue change"]
    )["store_id"]
]

focus_store_departments = (
    store_department_performance.reset_index()
    .loc[lambda data: data["STORE_ID"].isin(focus_stores)]
    .sort_values(["STORE_ID", "Change"])
)

print("\nFocus store breakdown — Revenue in USD")
display(focus_store_departments.round(2))

Department performance — Revenue in USD


,Department,Previous week,Current week,Change,Change (%)
2,Foods 1,71509.07,69255.87,-2253.20,-3.15
0,Foods 2,168175.53,130507.89,-37667.64,-22.40
1,Foods 3,373318.11,340031.34,-33286.77,-8.92



Focus store breakdown — Revenue in USD


,STORE_ID,DEPT_ID,Previous week,Current week,Change,Change (%)
4,CA_2,FOODS_2,13926.69,13946.00,19.31,0.14
3,CA_2,FOODS_1,9970.29,10715.11,744.82,7.47
5,CA_2,FOODS_3,36902.15,38457.53,1555.38,4.21
25,WI_2,FOODS_2,37462.51,20808.44,-16654.07,-44.46
26,WI_2,FOODS_3,48482.78,34341.53,-14141.25,-29.17
24,WI_2,FOODS_1,9560.76,7961.52,-1599.24,-16.73


In [39]:
import math

for store_id in focus_stores:
    breakdown = store_department_performance.loc[store_id]
    store_total = evidence_lookup[f"store_{store_id}"]

    for column, source_field in [
        ("Previous week", "Previous revenue"),
        ("Current week", "Current revenue"),
        ("Change", "Revenue change")
    ]:
        assert math.isclose(
            breakdown[column].sum(),
            store_total[source_field],
            rel_tol=0,
            abs_tol=0.01
        ), f"Reconciliation failed: {store_id}, {column}"

    print(f"{store_id}: department totals match store totals — PASS")

store_department_evidence = []

for _, row in focus_store_departments.iterrows():
    store_department_evidence.append({
        "evidence_id": (
            f"store_department_{row['STORE_ID']}_{row['DEPT_ID']}"
        ),
        "store_id": row["STORE_ID"],
        "department_id": row["DEPT_ID"],
        "previous_revenue": float(row["Previous week"]),
        "current_revenue": float(row["Current week"]),
        "revenue_change": float(row["Change"]),
        "wow_change_pct": (
            None if pd.isna(row["Change (%)"])
            else float(row["Change (%)"])
        )
    })

breakdown_file = revised_file.with_name(
    f"{revised_file.stem}_store_department_evidence.json"
)

breakdown_file.write_text(
    json.dumps(
        {
            "report_id": evidence_package["report_id"],
            "report_period": evidence_package["report_period"],
            "comparison_period": evidence_package["comparison_period"],
            "currency": "USD",
            "records": store_department_evidence
        },
        indent=2,
        allow_nan=False
    ),
    encoding="utf-8"
)

print("\nNew evidence records:", len(store_department_evidence))
print("Evidence saved:", breakdown_file)

WI_2: department totals match store totals — PASS
CA_2: department totals match store totals — PASS

New evidence records: 6
Evidence saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T0330056875Z_draft_revised_store_department_evidence.json


In [40]:
from IPython.display import Markdown, display


def display_label(identifier):
    if identifier.startswith("FOODS_"):
        return identifier.replace("FOODS_", "Foods ")
    return identifier.replace("_", " ")


def revenue_table(records, label_field):
    lines = [
        "| Department | Previous week | Current week | Change | Change (%) |",
        "| --- | ---: | ---: | ---: | ---: |"
    ]

    for item in records:
        pct = item["wow_change_pct"]
        pct_text = "N/A" if pct is None else f"{pct:+.2f}%"

        lines.append(
            f"| {display_label(item[label_field])} "
            f"| {item['previous_revenue']:,.2f} "
            f"| {item['current_revenue']:,.2f} "
            f"| {item['revenue_change']:+,.2f} "
            f"| {pct_text} |"
        )

    return "\n".join(lines)


report = json.loads(revised_file.read_text(encoding="utf-8"))
breakdown_data = json.loads(
    breakdown_file.read_text(encoding="utf-8")
)

units = evidence_lookup["overall_1"]
revenue = evidence_lookup["overall_2"]

department_records = [
    {
        "department_id": item["department_id"],
        "previous_revenue": item["Previous revenue"],
        "current_revenue": item["Current revenue"],
        "revenue_change": item["Revenue change"],
        "wow_change_pct": item["WoW change (%)"]
    }
    for item in sorted(
        evidence_package["department_metrics"],
        key=lambda item: item["department_id"]
    )
]

lines = [
    "# Weekly Sales Review",
    "",
    (
        f"**DRAFT | {report['report_period']['start']} "
        f"to {report['report_period']['end']}**"
    ),
    "",
    (
        f"Compared with {report['comparison_period']['start']} "
        f"to {report['comparison_period']['end']}."
    ),
    "",
    "Scope: 10 stores across CA, TX and WI; three food departments.",
    "All monetary values are in USD.",
    "",
    "## Overall Performance",
    "",
    "| Metric | Previous week | Current week | Change | Change (%) |",
    "| --- | ---: | ---: | ---: | ---: |",
    (
        f"| Revenue | {revenue['previous_value']:,.2f} "
        f"| {revenue['current_value']:,.2f} "
        f"| {revenue['change']:+,.2f} "
        f"| {revenue['wow_change_pct']:+.2f}% |"
    ),
    (
        f"| Units sold | {units['previous_value']:,.0f} "
        f"| {units['current_value']:,.0f} "
        f"| {units['change']:+,.0f} "
        f"| {units['wow_change_pct']:+.2f}% |"
    ),
    "",
    "## Department Performance",
    "",
    revenue_table(department_records, "department_id"),
    "",
    "## Store Highlights",
    ""
]

store_metrics = evidence_package["store_metrics"]
decline_store = min(store_metrics, key=lambda x: x["Revenue change"])
growth_store = max(store_metrics, key=lambda x: x["Revenue change"])

focus_records = {}

for store_item in [decline_store, growth_store]:
    store_id = store_item["store_id"]

    records = sorted(
        [
            item for item in breakdown_data["records"]
            if item["store_id"] == store_id
        ],
        key=lambda item: item["department_id"]
    )

    assert len(records) == 3, f"Incomplete breakdown: {store_id}"
    focus_records[store_id] = records

    lines.extend([
        f"### {display_label(store_id)}",
        "",
        (
            f"Revenue change: **{store_item['Revenue change']:+,.2f} "
            f"({store_item['WoW change (%)']:+.2f}%)**."
        ),
        "",
        revenue_table(records, "department_id"),
        ""
    ])

decline_departments = sorted(
    [
        item for item in focus_records[decline_store["store_id"]]
        if item["revenue_change"] < 0
    ],
    key=lambda item: item["revenue_change"]
)[:2]

growth_departments = sorted(
    [
        item for item in focus_records[growth_store["store_id"]]
        if item["revenue_change"] > 0
    ],
    key=lambda item: item["revenue_change"],
    reverse=True
)[:1]

action_items = []

for store_item, records in [
    (decline_store, decline_departments),
    (growth_store, growth_departments)
]:
    if not records:
        continue

    store_label = display_label(store_item["store_id"])
    department_labels = ", ".join(
        display_label(item["department_id"]) for item in records
    )

    action_items.append({
        "action_id": f"action_{len(action_items) + 1}",
        "action": (
            f"Compare daily revenue and units sold across both weeks "
            f"for {store_label}: {department_labels}."
        ),
        "expected_output": (
            "Identify the dates with the largest changes "
            "and whether revenue and units moved together."
        ),
        "evidence_ids": [
            item["evidence_id"] for item in records
        ]
    })

lines.extend([
    "## Proposed Actions",
    "",
    "| Action | Expected output |",
    "| --- | --- |"
])

for item in action_items:
    lines.append(
        f"| {item['action']} | {item['expected_output']} |"
    )

lines.extend([
    "",
    "*Historical data; causes are unconfirmed. "
    "Actions are proposed, not completed. Human approval is pending.*"
])

manager_preview = "\n".join(lines)

manager_preview_file = revised_file.with_name(
    f"{revised_file.stem}_manager_preview.md"
)

manager_preview_file.write_text(
    manager_preview,
    encoding="utf-8"
)

action_file = revised_file.with_name(
    f"{revised_file.stem}_proposed_actions.json"
)

action_file.write_text(
    json.dumps(
        {
            "report_id": report["report_id"],
            "status": "proposed",
            "source": "Rule-based review actions; not LLM-generated",
            "actions": action_items
        },
        indent=2,
        allow_nan=False
    ),
    encoding="utf-8"
)

display(Markdown(manager_preview))
print("Manager preview saved:", manager_preview_file)
print("Proposed actions saved:", action_file)

# Weekly Sales Review

**DRAFT | 2016-05-16 to 2016-05-22**

Compared with 2016-05-09 to 2016-05-15.

Scope: 10 stores across CA, TX and WI; three food departments.
All monetary values are in USD.

## Overall Performance

| Metric | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Revenue | 613,002.71 | 539,795.10 | -73,207.61 | -11.94% |
| Units sold | 225,812 | 202,535 | -23,277 | -10.31% |

## Department Performance

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 71,509.07 | 69,255.87 | -2,253.20 | -3.15% |
| Foods 2 | 168,175.53 | 130,507.89 | -37,667.64 | -22.40% |
| Foods 3 | 373,318.11 | 340,031.34 | -33,286.77 | -8.92% |

## Store Highlights

### WI 2

Revenue change: **-32,394.56 (-33.92%)**.

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 9,560.76 | 7,961.52 | -1,599.24 | -16.73% |
| Foods 2 | 37,462.51 | 20,808.44 | -16,654.07 | -44.46% |
| Foods 3 | 48,482.78 | 34,341.53 | -14,141.25 | -29.17% |

### CA 2

Revenue change: **+2,319.51 (+3.82%)**.

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 9,970.29 | 10,715.11 | +744.82 | +7.47% |
| Foods 2 | 13,926.69 | 13,946.00 | +19.31 | +0.14% |
| Foods 3 | 36,902.15 | 38,457.53 | +1,555.38 | +4.21% |

## Proposed Actions

| Action | Expected output |
| --- | --- |
| Compare daily revenue and units sold across both weeks for WI 2: Foods 2, Foods 3. | Identify the dates with the largest changes and whether revenue and units moved together. |
| Compare daily revenue and units sold across both weeks for CA 2: Foods 3. | Identify the dates with the largest changes and whether revenue and units moved together. |

*Historical data; causes are unconfirmed. Actions are proposed, not completed. Human approval is pending.*

Manager preview saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T0330056875Z_draft_revised_manager_preview.md
Proposed actions saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T0330056875Z_draft_revised_proposed_actions.json


In [41]:
processed_folder = project_folder / "data" / "processed"

file_names = [
    "future_sales_forecast_monthly_patterns.csv",
    "sales_anomaly_review_monthly_patterns.csv"
]

for file_name in file_names:
    file_path = processed_folder / file_name

    print(f"\nFile: {file_name}")

    if not file_path.exists():
        print("File not found.")
        continue

    sample = pd.read_csv(file_path, nrows=5)

    print("Columns:", sample.columns.tolist())
    display(sample)


File: future_sales_forecast_monthly_patterns.csv
Columns: ['forecast_origin', 'sales_date', 'state_id', 'store_id', 'dept_id', 'baseline_units', 'adjustment_factor', 'predicted_units', 'model']


,forecast_origin,sales_date,state_id,store_id,dept_id,baseline_units,adjustment_factor,predicted_units,model
0,2016-05-22,2016-05-23,CA,CA_1,FOODS_1,282.75,0.991252,280.276592,Calendar + monthly patterns
1,2016-05-22,2016-05-24,CA,CA_1,FOODS_1,304.75,0.991252,302.084142,Calendar + monthly patterns
2,2016-05-22,2016-05-25,CA,CA_1,FOODS_1,304.00,0.991252,301.340703,Calendar + monthly patterns
3,2016-05-22,2016-05-26,CA,CA_1,FOODS_1,306.25,0.991252,303.571021,Calendar + monthly patterns
4,2016-05-22,2016-05-27,CA,CA_1,FOODS_1,383.25,0.991252,379.897449,Calendar + monthly patterns



File: sales_anomaly_review_monthly_patterns.csv
Columns: ['sales_date', 'state_id', 'store_id', 'dept_id', 'actual_units', 'expected_units', 'deviation_units', 'deviation_pct', 'is_alert', 'alert_type', 'model', 'forecast_origin', 'percentage_threshold', 'units_threshold']


,sales_date,state_id,store_id,dept_id,actual_units,expected_units,deviation_units,deviation_pct,is_alert,alert_type,model,forecast_origin,percentage_threshold,units_threshold
0,2016-04-25,CA,CA_1,FOODS_1,294,253.761426,40.238574,15.856852,False,Within threshold,Calendar + monthly patterns,2016-04-24,25,100
1,2016-04-26,CA,CA_1,FOODS_1,259,220.501434,38.498566,17.459554,False,Within threshold,Calendar + monthly patterns,2016-04-24,25,100
2,2016-04-27,CA,CA_1,FOODS_1,268,228.878024,39.121976,17.092937,False,Within threshold,Calendar + monthly patterns,2016-04-24,25,100
3,2016-04-28,CA,CA_1,FOODS_1,328,294.166158,33.833842,11.501609,False,Within threshold,Calendar + monthly patterns,2016-04-24,25,100
4,2016-04-29,CA,CA_1,FOODS_1,387,311.165710,75.834290,24.371030,False,Within threshold,Calendar + monthly patterns,2016-04-24,25,100


In [42]:
forecast = pd.read_csv(
    processed_folder / "future_sales_forecast_monthly_patterns.csv"
)

anomaly_review = pd.read_csv(
    processed_folder / "sales_anomaly_review_monthly_patterns.csv"
)

for name, data, value_columns in [
    ("Forecast", forecast, ["predicted_units"]),
    ("Historical review", anomaly_review, ["actual_units", "expected_units"])
]:
    for column in ["sales_date", "forecast_origin"]:
        data[column] = pd.to_datetime(
            data[column], errors="raise"
        ).dt.normalize()

    key_columns = ["sales_date", "store_id", "dept_id"]

    print(f"\n{name}")
    print("Rows:", len(data))
    print("Start date:", data["sales_date"].min().date())
    print("End date:", data["sales_date"].max().date())
    print("Forecast origins:", data["forecast_origin"].unique())
    print("Models:", data["model"].unique())
    print("Duplicate keys:", data.duplicated(key_columns).sum())
    print(
        "Missing required values:",
        data[
            key_columns + ["forecast_origin", "model"] + value_columns
        ].isna().sum().sum()
    )
    print("Negative units:", (data[value_columns] < 0).sum().sum())
    print(
        "Origins on or after sales date:",
        (data["forecast_origin"] >= data["sales_date"]).sum()
    )

print("\nAlert flag values:")
print(anomaly_review["is_alert"].value_counts(dropna=False))


Forecast
Rows: 840
Start date: 2016-05-23
End date: 2016-06-19
Forecast origins: <DatetimeArray>
['2016-05-22 00:00:00']
Length: 1, dtype: datetime64[ns]
Models: ['Calendar + monthly patterns']
Duplicate keys: 0
Missing required values: 0
Negative units: 0
Origins on or after sales date: 0

Historical review
Rows: 840
Start date: 2016-04-25
End date: 2016-05-22
Forecast origins: <DatetimeArray>
['2016-04-24 00:00:00']
Length: 1, dtype: datetime64[ns]
Models: ['Calendar + monthly patterns']
Duplicate keys: 0
Missing required values: 0
Negative units: 0
Origins on or after sales date: 0

Alert flag values:
is_alert
False    748
True      92
Name: count, dtype: int64


In [43]:
report_start = pd.Timestamp(evidence_package["report_period"]["start"])
report_end = pd.Timestamp(evidence_package["report_period"]["end"])

weekly_alert_review = anomaly_review.loc[
    anomaly_review["sales_date"].between(report_start, report_end)
].copy()

weekly_actual_sales = sales.loc[
    sales["SALES_DATE"].between(report_start, report_end),
    ["SALES_DATE", "STORE_ID", "DEPT_ID", "TOTAL_UNITS_SOLD"]
].rename(
    columns={
        "SALES_DATE": "sales_date",
        "STORE_ID": "store_id",
        "DEPT_ID": "dept_id",
        "TOTAL_UNITS_SOLD": "source_actual_units",
    }
)

comparison = weekly_alert_review.merge(
    weekly_actual_sales,
    on=["sales_date", "store_id", "dept_id"],
    how="outer",
    validate="one_to_one",
    indicator=True,
)

assert len(comparison) == 210, "Unexpected weekly record count."
assert comparison["_merge"].eq("both").all(), "Sales records do not align."

unit_difference = (
    comparison["actual_units"] - comparison["source_actual_units"]
).abs()

assert unit_difference.le(0.000001).all(), "Actual units do not match."

print(f"Report period: {report_start.date()} to {report_end.date()}")
print(f"Weekly records: {len(weekly_alert_review)}")
print("Actual units reconciliation: PASS")

print("\nWeekly alert flag values:")
print(weekly_alert_review["is_alert"].value_counts(dropna=False))

print("\nWeekly alert types:")
print(weekly_alert_review["alert_type"].value_counts(dropna=False))

Report period: 2016-05-16 to 2016-05-22
Weekly records: 210
Actual units reconciliation: PASS

Weekly alert flag values:
is_alert
False    192
True      18
Name: count, dtype: int64

Weekly alert types:
alert_type
Within threshold    192
Above expected       15
Below expected        3
Name: count, dtype: int64


In [44]:
import numpy as np

review = weekly_alert_review.copy()

numeric_columns = [
    "actual_units",
    "expected_units",
    "deviation_units",
    "deviation_pct",
]

assert np.isfinite(
    review[numeric_columns].to_numpy(dtype=float)
).all(), "Non-finite numerical values found."

assert review["expected_units"].gt(0).all(), (
    "Zero or negative expected units require separate handling."
)

calculated_deviation = review["actual_units"] - review["expected_units"]
calculated_pct = calculated_deviation / review["expected_units"] * 100

assert np.allclose(
    review["deviation_units"],
    calculated_deviation,
    rtol=0,
    atol=0.000001,
), "Deviation units do not match."

assert np.allclose(
    review["deviation_pct"],
    calculated_pct,
    rtol=0,
    atol=0.000001,
), "Deviation percentages do not match."

calculated_alert = (
    calculated_pct.abs().ge(25)
    & calculated_deviation.abs().ge(100)
)

flag_text = review["is_alert"].astype(str).str.strip().str.lower()

assert flag_text.isin(["true", "false"]).all(), (
    "Unexpected alert flag values."
)

recorded_alert = flag_text.eq("true")

assert recorded_alert.eq(calculated_alert).all(), (
    "Recorded alert flags do not match the thresholds."
)

calculated_type = np.select(
    [
        calculated_alert & calculated_deviation.gt(0),
        calculated_alert & calculated_deviation.lt(0),
    ],
    ["Above expected", "Below expected"],
    default="Within threshold",
)

assert review["alert_type"].eq(calculated_type).all(), (
    "Recorded alert types do not match."
)

weekly_alerts = review.loc[calculated_alert].copy()

print("Deviation calculations: PASS")
print("Alert thresholds: PASS")
print("Alert classifications: PASS")
print(f"Verified weekly alerts: {len(weekly_alerts)}")

Deviation calculations: PASS
Alert thresholds: PASS
Alert classifications: PASS
Verified weekly alerts: 18


In [45]:
top_weekly_alerts = (
    weekly_alerts
    .assign(
        absolute_deviation=lambda data: data["deviation_units"].abs()
    )
    .sort_values(
        ["absolute_deviation", "sales_date", "store_id", "dept_id"],
        ascending=[False, True, True, True],
    )
    .head(3)
    .copy()
)

alert_preview = top_weekly_alerts[
    [
        "sales_date",
        "store_id",
        "dept_id",
        "actual_units",
        "expected_units",
        "deviation_units",
        "deviation_pct",
        "alert_type",
    ]
].copy()

alert_preview["sales_date"] = (
    alert_preview["sales_date"].dt.strftime("%Y-%m-%d")
)
alert_preview["store_id"] = (
    alert_preview["store_id"].str.replace("_", " ", regex=False)
)
alert_preview["dept_id"] = (
    alert_preview["dept_id"]
    .str.replace("FOODS_", "Foods ", regex=False)
)

alert_preview.columns = [
    "Date",
    "Store",
    "Department",
    "Actual units",
    "Expected units",
    "Difference",
    "Deviation (%)",
    "Alert type",
]

display(
    alert_preview.style.format(
        {
            "Actual units": "{:,.0f}",
            "Expected units": "{:,.1f}",
            "Difference": "{:+,.1f}",
            "Deviation (%)": "{:+.1f}%",
        }
    ).hide(axis="index")
)

print(f"Total weekly alerts: {len(weekly_alerts)}")
print(
    "Selection rule: Top 3 alerts by absolute unit difference."
)
print(
    "Historical exploratory alerts; causes require investigation."
)

Date,Store,Department,Actual units,Expected units,Difference,Deviation (%),Alert type
2016-05-21,WI 2,Foods 2,850,"1,319.7",-469.7,-35.6%,Below expected
2016-05-20,WI 2,Foods 2,815,"1,188.8",-373.8,-31.4%,Below expected
2016-05-16,WI 1,Foods 3,"1,686","1,312.4",+373.6,+28.5%,Above expected


Total weekly alerts: 18
Selection rule: Top 3 alerts by absolute unit difference.
Historical exploratory alerts; causes require investigation.


In [46]:
forecast_start = report_end + pd.Timedelta(days=1)
forecast_end = forecast_start + pd.Timedelta(days=6)

next_week_forecast = forecast.loc[
    forecast["sales_date"].between(forecast_start, forecast_end)
].copy()

key_columns = ["sales_date", "store_id", "dept_id"]

expected_keys = pd.MultiIndex.from_product(
    [
        pd.date_range(forecast_start, forecast_end),
        sorted(weekly_actual_sales["store_id"].unique()),
        sorted(weekly_actual_sales["dept_id"].unique()),
    ],
    names=key_columns,
)

actual_keys = pd.MultiIndex.from_frame(
    next_week_forecast[key_columns]
)

assert not actual_keys.has_duplicates, "Duplicate forecast records."
assert expected_keys.difference(actual_keys).empty, (
    "Missing forecast records."
)
assert actual_keys.difference(expected_keys).empty, (
    "Unexpected forecast records."
)
assert np.isfinite(
    next_week_forecast["predicted_units"].to_numpy(dtype=float)
).all(), "Invalid forecast values."
assert next_week_forecast["predicted_units"].ge(0).all(), (
    "Negative forecast values."
)
assert next_week_forecast["forecast_origin"].eq(report_end).all(), (
    "Unexpected forecast origin."
)

forecast_by_department = (
    next_week_forecast.groupby("dept_id")["predicted_units"]
    .sum()
    .rename("Forecast units")
    .to_frame()
)

forecast_preview = forecast_by_department.copy()
forecast_preview.index = (
    forecast_preview.index.str.replace(
        "FOODS_", "Foods ", regex=False
    )
)
forecast_preview.index.name = "Department"

print(
    f"Forecast period: {forecast_start.date()} "
    f"to {forecast_end.date()}"
)
print(f"Forecast records: {len(next_week_forecast)}")
print("Forecast coverage and values: PASS")

display(
    forecast_preview.style.format(
        {"Forecast units": "{:,.0f}"}
    )
)

print(
    "Total forecast units: "
    f"{next_week_forecast['predicted_units'].sum():,.0f}"
)
print("Model: Calendar + monthly patterns")
print("Forecasts are estimates, not confirmed future sales.")

Forecast period: 2016-05-23 to 2016-05-29
Forecast records: 210
Forecast coverage and values: PASS


,Forecast units
Department,
Foods 1,"24,744"
Foods 2,"35,567"
Foods 3,"130,820"


Total forecast units: 191,131
Model: Calendar + monthly patterns
Forecasts are estimates, not confirmed future sales.


In [47]:
from IPython.display import Markdown, display

alert_lines = [
    "## Historical Sales Alerts",
    "",
    (
        f"{len(weekly_alerts)} daily store-department records "
        f"met both thresholds: absolute deviation of at least "
        f"25% and at least 100 units."
    ),
    "",
    (
        f"Above expected: "
        f"{weekly_alerts['alert_type'].eq('Above expected').sum()}; "
        f"below expected: "
        f"{weekly_alerts['alert_type'].eq('Below expected').sum()}."
    ),
    "",
    "Top 3 records by absolute unit difference:",
    "",
    "| Date | Store | Department | Actual units | Expected units | Difference | Deviation |",
    "| --- | --- | --- | ---: | ---: | ---: | ---: |",
]

for _, row in top_weekly_alerts.iterrows():
    alert_lines.append(
        f"| {row['sales_date']:%Y-%m-%d} "
        f"| {display_label(row['store_id'])} "
        f"| {display_label(row['dept_id'])} "
        f"| {row['actual_units']:,.0f} "
        f"| {row['expected_units']:,.1f} "
        f"| {row['deviation_units']:+,.1f} "
        f"| {row['deviation_pct']:+.1f}% |"
    )

alert_lines.extend([
    "",
    (
        "These historical alerts are exploratory. "
        "They identify records for investigation, "
        "not confirmed business issues."
    ),
])

forecast_lines = [
    "## Next-Week Sales Forecast",
    "",
    f"Forecast period: {forecast_start:%Y-%m-%d} to {forecast_end:%Y-%m-%d}.",
    "",
    "| Department | Forecast units |",
    "| --- | ---: |",
]

for department, row in forecast_by_department.iterrows():
    forecast_lines.append(
        f"| {display_label(department)} "
        f"| {row['Forecast units']:,.0f} |"
    )

forecast_lines.extend([
    (
        f"| **Total** "
        f"| **{next_week_forecast['predicted_units'].sum():,.0f}** |"
    ),
    "",
    (
        "Model: Calendar + monthly patterns. "
        "Values are rounded for display. "
        "Forecasts are estimates, not confirmed future sales."
    ),
])

new_sections = "\n".join(
    alert_lines + [""] + forecast_lines
)

base_preview = manager_preview_file.read_text(encoding="utf-8")
action_heading = "## Proposed Actions"

assert action_heading in base_preview, (
    "Proposed Actions heading not found. Check the preview headings."
)

if "## Historical Sales Alerts" in base_preview:
    section_start = base_preview.index("## Historical Sales Alerts")
    action_start = base_preview.index(action_heading)
    assert section_start < action_start
    base_preview = (
        base_preview[:section_start]
        + base_preview[action_start:]
    )

updated_manager_preview = base_preview.replace(
    action_heading,
    new_sections + "\n\n" + action_heading,
    1,
)

updated_manager_file = manager_preview_file.with_name(
    manager_preview_file.stem + "_with_forecast_alerts.md"
)

updated_manager_file.write_text(
    updated_manager_preview,
    encoding="utf-8",
)

display(Markdown(updated_manager_preview))
print(f"Updated manager preview saved: {updated_manager_file}")
print("Human approval: PENDING")

# Weekly Sales Review

**DRAFT | 2016-05-16 to 2016-05-22**

Compared with 2016-05-09 to 2016-05-15.

Scope: 10 stores across CA, TX and WI; three food departments.
All monetary values are in USD.

## Overall Performance

| Metric | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Revenue | 613,002.71 | 539,795.10 | -73,207.61 | -11.94% |
| Units sold | 225,812 | 202,535 | -23,277 | -10.31% |

## Department Performance

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 71,509.07 | 69,255.87 | -2,253.20 | -3.15% |
| Foods 2 | 168,175.53 | 130,507.89 | -37,667.64 | -22.40% |
| Foods 3 | 373,318.11 | 340,031.34 | -33,286.77 | -8.92% |

## Store Highlights

### WI 2

Revenue change: **-32,394.56 (-33.92%)**.

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 9,560.76 | 7,961.52 | -1,599.24 | -16.73% |
| Foods 2 | 37,462.51 | 20,808.44 | -16,654.07 | -44.46% |
| Foods 3 | 48,482.78 | 34,341.53 | -14,141.25 | -29.17% |

### CA 2

Revenue change: **+2,319.51 (+3.82%)**.

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 9,970.29 | 10,715.11 | +744.82 | +7.47% |
| Foods 2 | 13,926.69 | 13,946.00 | +19.31 | +0.14% |
| Foods 3 | 36,902.15 | 38,457.53 | +1,555.38 | +4.21% |

## Historical Sales Alerts

18 daily store-department records met both thresholds: absolute deviation of at least 25% and at least 100 units.

Above expected: 15; below expected: 3.

Top 3 records by absolute unit difference:

| Date | Store | Department | Actual units | Expected units | Difference | Deviation |
| --- | --- | --- | ---: | ---: | ---: | ---: |
| 2016-05-21 | WI 2 | Foods 2 | 850 | 1,319.7 | -469.7 | -35.6% |
| 2016-05-20 | WI 2 | Foods 2 | 815 | 1,188.8 | -373.8 | -31.4% |
| 2016-05-16 | WI 1 | Foods 3 | 1,686 | 1,312.4 | +373.6 | +28.5% |

These historical alerts are exploratory. They identify records for investigation, not confirmed business issues.

## Next-Week Sales Forecast

Forecast period: 2016-05-23 to 2016-05-29.

| Department | Forecast units |
| --- | ---: |
| Foods 1 | 24,744 |
| Foods 2 | 35,567 |
| Foods 3 | 130,820 |
| **Total** | **191,131** |

Model: Calendar + monthly patterns. Values are rounded for display. Forecasts are estimates, not confirmed future sales.

## Proposed Actions

| Action | Expected output |
| --- | --- |
| Compare daily revenue and units sold across both weeks for WI 2: Foods 2, Foods 3. | Identify the dates with the largest changes and whether revenue and units moved together. |
| Compare daily revenue and units sold across both weeks for CA 2: Foods 3. | Identify the dates with the largest changes and whether revenue and units moved together. |

*Historical data; causes are unconfirmed. Actions are proposed, not completed. Human approval is pending.*

Updated manager preview saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T0330056875Z_draft_revised_manager_preview_with_forecast_alerts.md
Human approval: PENDING


In [48]:
import json

alert_records = []

for _, row in weekly_alerts.iterrows():
    alert_records.append({
        "evidence_id": (
            f"alert_{row['sales_date']:%Y%m%d}_"
            f"{row['store_id']}_{row['dept_id']}"
        ),
        "sales_date": row["sales_date"].strftime("%Y-%m-%d"),
        "store_id": row["store_id"],
        "department_id": row["dept_id"],
        "actual_units": float(row["actual_units"]),
        "expected_units": float(row["expected_units"]),
        "deviation_units": float(row["deviation_units"]),
        "deviation_pct": float(row["deviation_pct"]),
        "alert_type": row["alert_type"],
    })

model_evidence = {
    "schema_version": "1.0",
    "report_id": evidence_package["report_id"],
    "model": "Calendar + monthly patterns",
    "sources": {
        "alerts": (
            "data/processed/"
            "sales_anomaly_review_monthly_patterns.csv"
        ),
        "forecast": (
            "data/processed/"
            "future_sales_forecast_monthly_patterns.csv"
        ),
    },
    "alert_summary": {
        "evidence_id": "weekly_alert_summary",
        "report_period": evidence_package["report_period"],
        "forecast_origin": "2016-04-24",
        "reviewed_records": int(len(weekly_alert_review)),
        "alert_count": int(len(weekly_alerts)),
        "above_expected": int(
            weekly_alerts["alert_type"].eq("Above expected").sum()
        ),
        "below_expected": int(
            weekly_alerts["alert_type"].eq("Below expected").sum()
        ),
        "thresholds": {
            "absolute_deviation_pct_at_least": 25,
            "absolute_deviation_units_at_least": 100,
            "operator": "AND",
        },
    },
    "alert_records": alert_records,
    "forecast_summary": {
        "evidence_id": "next_week_forecast_total",
        "forecast_origin": report_end.strftime("%Y-%m-%d"),
        "period_start": forecast_start.strftime("%Y-%m-%d"),
        "period_end": forecast_end.strftime("%Y-%m-%d"),
        "record_count": int(len(next_week_forecast)),
        "predicted_units": float(
            next_week_forecast["predicted_units"].sum()
        ),
    },
    "department_forecasts": [
        {
            "evidence_id": f"forecast_{department}",
            "department_id": department,
            "predicted_units": float(row["Forecast units"]),
        }
        for department, row in forecast_by_department.iterrows()
    ],
    "limitations": [
        "Historical alerts are exploratory.",
        "Alerts do not establish causes or confirmed business issues.",
        "Forecasts are estimates of units sold, not revenue.",
        "Model training and tuning leakage have not been audited here.",
        "Human review is required before approval.",
    ],
}

model_evidence_file = (
    processed_folder.parent.parent
    / "reports"
    / "evidence"
    / f"{evidence_package['report_id']}_model_evidence.json"
)

model_evidence_file.parent.mkdir(parents=True, exist_ok=True)

model_evidence_file.write_text(
    json.dumps(model_evidence, indent=2, allow_nan=False),
    encoding="utf-8",
)

saved_evidence = json.loads(
    model_evidence_file.read_text(encoding="utf-8")
)
assert saved_evidence == model_evidence, "Saved evidence differs."

print(f"Alert evidence records: {len(alert_records)}")
print("Evidence save and read-back: PASS")
print(f"Saved: {model_evidence_file}")

Alert evidence records: 18
Evidence save and read-back: PASS
Saved: D:\ai-fmcg-sales-insights\reports\evidence\weekly_sales_20160522_model_evidence.json


In [49]:
from copy import deepcopy
import json

full_evidence = deepcopy(evidence_package)
full_evidence["schema_version"] = "1.1"

breakdown_evidence = json.loads(
    breakdown_file.read_text(encoding="utf-8")
)

assert (
    breakdown_evidence["report_id"]
    == full_evidence["report_id"]
    == model_evidence["report_id"]
), "Report IDs do not match."

full_evidence["store_department_metrics"] = (
    breakdown_evidence["records"]
)
full_evidence["model_evidence"] = deepcopy(model_evidence)

outdated_limitation = (
    "Forecasts and model-based alerts are not included yet."
)

full_evidence["limitations"] = [
    item
    for item in full_evidence["limitations"]
    if item != outdated_limitation
]

for item in model_evidence["limitations"]:
    if item not in full_evidence["limitations"]:
        full_evidence["limitations"].append(item)

full_evidence_file = model_evidence_file.with_name(
    f"{full_evidence['report_id']}_full_evidence_v1.json"
)

full_evidence_file.write_text(
    json.dumps(full_evidence, indent=2, allow_nan=False),
    encoding="utf-8",
)

assert json.loads(
    full_evidence_file.read_text(encoding="utf-8")
) == full_evidence, "Saved evidence differs."

print("Complete evidence package: PASS")
print(
    "Store-department records:",
    len(full_evidence["store_department_metrics"]),
)
print(
    "Alert records:",
    len(full_evidence["model_evidence"]["alert_records"]),
)
print(f"Saved: {full_evidence_file}")

Complete evidence package: PASS
Store-department records: 6
Alert records: 18
Saved: D:\ai-fmcg-sales-insights\reports\evidence\weekly_sales_20160522_full_evidence_v1.json


In [50]:
report_instructions_v2 = """
Prepare a concise weekly sales briefing for a sales operations manager.
Use only the supplied evidence package.

Rules:
1. Write in clear business English.
2. Copy report_id, report_period and comparison_period exactly.
3. Set status to "draft".
4. Support every factual statement and proposed action with evidence_ids.
5. Use only numbers supported by the cited evidence.
6. Round revenue to two decimals, units to whole numbers, and percentages
   to two decimals. Label monetary amounts as USD.
7. Display identifiers as readable labels: WI_2 as WI 2 and FOODS_2 as Foods 2.
   Keep evidence_ids unchanged.
8. Distinguish revenue, actual units, expected units and forecast units.
9. Keep week-over-week comparisons separate from model-based deviations.
10. Describe historical alerts as exploratory records for investigation.
    Do not call them confirmed business problems.
11. Describe forecasts as estimates, not guaranteed future sales.
12. Do not infer causes, stockouts, promotions, profitability or price changes.
13. Do not add store and department changes together; they overlap.
14. Avoid ranking claims such as "largest" unless all comparison records
    are cited.
15. Proposed actions must specify what to examine and the expected output.
    Do not invent staff assignments, deadlines or completed investigations.
16. Do not claim that model training or tuning leakage has been audited.
17. Keep narrative text across all sections under 250 words.
18. Return valid JSON only, without Markdown fences.

Required structure:
{
  "report_id": "Copy from evidence",
  "status": "draft",
  "report_period": {"start": "Copy from evidence", "end": "Copy from evidence"},
  "comparison_period": {"start": "Copy from evidence", "end": "Copy from evidence"},
  "executive_summary": [
    {"text": "Brief factual statement", "evidence_ids": ["relevant_id"]}
  ],
  "alert_commentary": [
    {"text": "Historical alert observation", "evidence_ids": ["relevant_id"]}
  ],
  "forecast_commentary": [
    {"text": "Forecast estimate with its period", "evidence_ids": ["relevant_id"]}
  ],
  "proposed_actions": [
    {
      "action": "Specific investigation",
      "expected_output": "What the investigation should produce",
      "evidence_ids": ["relevant_id"]
    }
  ],
  "limitations": [
    "Historical data; causes are unconfirmed.",
    "Historical alerts are exploratory.",
    "Forecasts are estimates.",
    "Human approval is pending."
  ]
}

Include:
- Exactly two executive summary statements: overall performance and
  a store or department observation.
- One or two historical alert observations.
- One forecast statement including the forecast period and total units.
- Exactly two proposed investigation actions.
"""

prompt_file_v2 = (
    processed_folder.parent.parent
    / "prompts"
    / "weekly_sales_review_v2.txt"
)

prompt_file_v2.parent.mkdir(parents=True, exist_ok=True)
prompt_file_v2.write_text(
    report_instructions_v2.strip(),
    encoding="utf-8",
)

print("Updated prompt saved.")
print(f"Saved: {prompt_file_v2}")
print("API call made: No")

Updated prompt saved.
Saved: D:\ai-fmcg-sales-insights\prompts\weekly_sales_review_v2.txt
API call made: No


In [51]:
import json
from datetime import datetime, timezone

evidence_for_ai = json.loads(
    full_evidence_file.read_text(encoding="utf-8")
)
instructions_for_ai = prompt_file_v2.read_text(encoding="utf-8")

response_v2 = client.responses.create(
    model="gpt-4.1-mini",
    instructions=instructions_for_ai,
    input=(
        "Generate the weekly sales briefing as a JSON object "
        "using only this evidence package:\n\n"
        + json.dumps(evidence_for_ai, allow_nan=False)
    ),
    text={"format": {"type": "json_object"}},
    max_output_tokens=2500,
    store=False,
)

print(f"Response status: {response_v2.status}")

run_id_v2 = datetime.now(timezone.utc).strftime(
    "%Y%m%dT%H%M%S%fZ"
)

draft_folder_v2 = full_evidence_file.parent.parent / "drafts"
draft_folder_v2.mkdir(parents=True, exist_ok=True)

raw_response_file_v2 = draft_folder_v2 / (
    f"{evidence_for_ai['report_id']}_{run_id_v2}_ai_v2_raw.txt"
)
raw_response_file_v2.write_text(
    response_v2.output_text,
    encoding="utf-8",
)

if response_v2.status != "completed":
    raise RuntimeError(
        "Response incomplete. Raw output saved; review before retrying."
    )

ai_draft_v2 = json.loads(response_v2.output_text)

ai_draft_file_v2 = raw_response_file_v2.with_suffix(".json")
ai_draft_file_v2.write_text(
    json.dumps(ai_draft_v2, indent=2, allow_nan=False),
    encoding="utf-8",
)

print(f"Draft saved: {ai_draft_file_v2}")
print("\nGenerated draft:")
print(json.dumps(ai_draft_v2, indent=2))
print("\nValidation: PENDING")
print("Human approval: PENDING")

Response status: completed
Draft saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T040027951984Z_ai_v2_raw.json

Generated draft:
{
  "report_id": "weekly_sales_20160522",
  "status": "draft",
  "report_period": {
    "start": "2016-05-16",
    "end": "2016-05-22"
  },
  "comparison_period": {
    "start": "2016-05-09",
    "end": "2016-05-15"
  },
  "executive_summary": [
    {
      "text": "Units sold decreased by 23,277 to 202,535 units, a 10.31% decline week-over-week, while sales revenue declined by USD 73,207.61 to USD 539,795.10, an 11.94% decrease.",
      "evidence_ids": [
        "overall_1",
        "overall_2"
      ]
    },
    {
      "text": "Store WI 2 experienced a revenue decline of USD 32,394.56, a 33.92% drop compared to the prior week.",
      "evidence_ids": [
        "store_WI_2"
      ]
    }
  ],
  "alert_commentary": [
    {
      "text": "Multiple alerts identified significant above-expected sales in Foods 1 at store CA 2 on May 

In [52]:
alert_evidence_lookup = {
    record["evidence_id"]: record
    for record in evidence_for_ai["model_evidence"]["alert_records"]
}

referenced_alert_ids = list(dict.fromkeys(
    evidence_id
    for item in ai_draft_v2["alert_commentary"]
    for evidence_id in item["evidence_ids"]
))

missing_ids = [
    evidence_id
    for evidence_id in referenced_alert_ids
    if evidence_id not in alert_evidence_lookup
]

assert not missing_ids, (
    f"Unknown alert evidence IDs: {missing_ids}"
)

referenced_alerts = pd.DataFrame([
    alert_evidence_lookup[evidence_id]
    for evidence_id in referenced_alert_ids
])

display(
    referenced_alerts[
        [
            "sales_date",
            "store_id",
            "department_id",
            "actual_units",
            "expected_units",
            "deviation_units",
            "deviation_pct",
        ]
    ].style.format({
        "actual_units": "{:,.0f}",
        "expected_units": "{:,.2f}",
        "deviation_units": "{:+,.2f}",
        "deviation_pct": "{:+.2f}%",
    }).hide(axis="index")
)

print("Evidence references: FOUND")
print("Numerical and semantic validation: PENDING")
print("Human approval: PENDING")

sales_date,store_id,department_id,actual_units,expected_units,deviation_units,deviation_pct
2016-05-21,CA_2,FOODS_1,809,581.00,+228.00,+39.24%
2016-05-22,CA_2,FOODS_1,735,587.66,+147.34,+25.07%
2016-05-20,WI_2,FOODS_2,815,"1,188.83",-373.83,-31.45%
2016-05-21,WI_2,FOODS_2,850,"1,319.66",-469.66,-35.59%
2016-05-22,WI_2,FOODS_2,836,"1,196.49",-360.49,-30.13%


Evidence references: FOUND
Numerical and semantic validation: PENDING
Human approval: PENDING


In [53]:
from copy import deepcopy
import json

revised_ai_draft_v2 = deepcopy(ai_draft_v2)

wi_alert_ids = [
    "alert_20160520_WI_2_FOODS_2",
    "alert_20160521_WI_2_FOODS_2",
    "alert_20160522_WI_2_FOODS_2",
]

wi_records = [
    alert_evidence_lookup[evidence_id]
    for evidence_id in wi_alert_ids
]

minimum_actual = min(row["actual_units"] for row in wi_records)
maximum_actual = max(row["actual_units"] for row in wi_records)

minimum_pct = min(abs(row["deviation_pct"]) for row in wi_records)
maximum_pct = max(abs(row["deviation_pct"]) for row in wi_records)

revised_ai_draft_v2["alert_commentary"][1] = {
    "text": (
        "Exploratory below-expected sales alerts occurred for "
        "Foods 2 at store WI 2 from 2016-05-20 to 2016-05-22. "
        f"Actual sales ranged from {minimum_actual:,.0f} "
        f"to {maximum_actual:,.0f} units, "
        f"{minimum_pct:.2f}% to {maximum_pct:.2f}% "
        "below expected levels."
    ),
    "evidence_ids": wi_alert_ids,
}

revised_ai_draft_v2["proposed_actions"][1] = {
    "action": (
        "Compare actual and expected daily units for Foods 1 "
        "at store CA 2 on 2016-05-21 and 2016-05-22, "
        "and examine adjacent dates in the historical review."
    ),
    "expected_output": (
        "A daily comparison showing whether above-expected "
        "sales were isolated to those dates or also appeared "
        "on adjacent dates; causes remain unconfirmed."
    ),
    "evidence_ids": [
        "alert_20160521_CA_2_FOODS_1",
        "alert_20160522_CA_2_FOODS_1",
    ],
}

revised_ai_draft_v2["revision"] = {
    "source_draft": ai_draft_file_v2.name,
    "method": "Programmatic edits following evidence review",
    "changes": [
        "Corrected the WI 2 Foods 2 actual-unit range.",
        "Replaced the inventory and staffing investigation "
        "with a daily unit comparison using available data.",
    ],
}

revised_ai_file_v2 = ai_draft_file_v2.with_name(
    ai_draft_file_v2.stem + "_revised.json"
)

revised_ai_file_v2.write_text(
    json.dumps(revised_ai_draft_v2, indent=2, allow_nan=False),
    encoding="utf-8",
)

print(revised_ai_draft_v2["alert_commentary"][1]["text"])
print(f"\nRevised draft saved: {revised_ai_file_v2}")
print("Remaining validation: PENDING")
print("Human approval: PENDING")

Exploratory below-expected sales alerts occurred for Foods 2 at store WI 2 from 2016-05-20 to 2016-05-22. Actual sales ranged from 815 to 850 units, 30.13% to 35.59% below expected levels.

Revised draft saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T040027951984Z_ai_v2_raw_revised.json
Remaining validation: PENDING
Human approval: PENDING


In [54]:
import json
from datetime import datetime, timezone

def collect_evidence_ids(value):
    found = []

    if isinstance(value, dict):
        if "evidence_id" in value:
            found.append(value["evidence_id"])
        for child in value.values():
            found.extend(collect_evidence_ids(child))

    elif isinstance(value, list):
        for child in value:
            found.extend(collect_evidence_ids(child))

    return found


all_evidence_ids = collect_evidence_ids(evidence_for_ai)

assert len(all_evidence_ids) == len(set(all_evidence_ids)), (
    "Duplicate evidence IDs found."
)

valid_ids_v2 = set(all_evidence_ids)
validation_errors_v2 = []

for field in ["report_id", "report_period", "comparison_period"]:
    if revised_ai_draft_v2.get(field) != evidence_for_ai[field]:
        validation_errors_v2.append(f"Incorrect {field}.")

if revised_ai_draft_v2.get("status") != "draft":
    validation_errors_v2.append("Status must be draft.")

section_rules = {
    "executive_summary": (2, 2, ["text"]),
    "alert_commentary": (1, 2, ["text"]),
    "forecast_commentary": (1, 1, ["text"]),
    "proposed_actions": (2, 2, ["action", "expected_output"]),
}

for section, (minimum, maximum, text_fields) in section_rules.items():
    items = revised_ai_draft_v2.get(section)

    if not isinstance(items, list):
        validation_errors_v2.append(f"{section}: expected a list.")
        continue

    if not minimum <= len(items) <= maximum:
        validation_errors_v2.append(f"{section}: incorrect item count.")

    for position, item in enumerate(items, start=1):
        label = f"{section}[{position}]"

        if not isinstance(item, dict):
            validation_errors_v2.append(f"{label}: expected an object.")
            continue

        for field in text_fields:
            text = item.get(field)
            if not isinstance(text, str) or not text.strip():
                validation_errors_v2.append(f"{label}: invalid {field}.")

        references = item.get("evidence_ids")
        if (
            not isinstance(references, list)
            or not references
            or not all(isinstance(ref, str) for ref in references)
        ):
            validation_errors_v2.append(f"{label}: invalid references.")
        elif not set(references).issubset(valid_ids_v2):
            validation_errors_v2.append(f"{label}: unknown evidence ID.")

limitations = revised_ai_draft_v2.get("limitations")
if (
    not isinstance(limitations, list)
    or not limitations
    or not all(
        isinstance(item, str) and item.strip()
        for item in limitations
    )
):
    validation_errors_v2.append("Invalid limitations.")

validation_record_v2 = {
    "report_id": evidence_for_ai["report_id"],
    "checked_at": datetime.now(timezone.utc).isoformat(),
    "reviewed_draft": revised_ai_file_v2.name,
    "basic_validation": (
        "PASS" if not validation_errors_v2 else "FAIL"
    ),
    "errors": validation_errors_v2,
    "original_draft_findings": [
        {
            "category": "incorrect_numeric_range",
            "location": "alert_commentary[2]",
            "original": "815 to 836 units",
            "corrected": "815 to 850 units",
            "detection_method": "Manual comparison with cited evidence",
        },
        {
            "category": "action_data_availability",
            "location": "proposed_actions[2]",
            "finding": (
                "The proposed investigation required inventory and "
                "staffing data not supplied in this project."
            ),
            "resolution": (
                "Replaced with a daily actual-versus-expected "
                "unit comparison using available data."
            ),
        },
    ],
    "scope": (
        "Checks structure, required text fields, report metadata "
        "and evidence ID existence. Does not establish that every "
        "claim is supported by its cited evidence."
    ),
    "remaining_numerical_and_semantic_review": "PENDING",
    "human_approval": "PENDING",
}

validation_folder_v2 = full_evidence_file.parent.parent / "validation"
validation_folder_v2.mkdir(parents=True, exist_ok=True)

validation_file_v2 = validation_folder_v2 / (
    revised_ai_file_v2.stem + "_validation.json"
)

validation_file_v2.write_text(
    json.dumps(validation_record_v2, indent=2, allow_nan=False),
    encoding="utf-8",
)

print(
    "Structure, metadata and evidence references:",
    validation_record_v2["basic_validation"],
)

for error in validation_errors_v2:
    print(f"- {error}")

print(f"Validation record saved: {validation_file_v2}")
print("Numerical and semantic review: PENDING")
print("Human approval: PENDING")

Structure, metadata and evidence references: PASS
Validation record saved: D:\ai-fmcg-sales-insights\reports\validation\weekly_sales_20160522_20261021T040027951984Z_ai_v2_raw_revised_validation.json
Numerical and semantic review: PENDING
Human approval: PENDING


In [55]:
def build_evidence_lookup(value):
    lookup = {}

    if isinstance(value, dict):
        if "evidence_id" in value:
            lookup[value["evidence_id"]] = value

        for child in value.values():
            lookup.update(build_evidence_lookup(child))

    elif isinstance(value, list):
        for child in value:
            lookup.update(build_evidence_lookup(child))

    return lookup


full_evidence_lookup = build_evidence_lookup(evidence_for_ai)

for section in [
    "executive_summary",
    "alert_commentary",
    "forecast_commentary",
    "proposed_actions",
]:
    print(f"\n{'=' * 60}\n{section.upper()}")

    for position, item in enumerate(
        revised_ai_draft_v2[section], start=1
    ):
        print(f"\nItem {position}:")

        if section == "proposed_actions":
            print("Action:", item["action"])
            print("Expected output:", item["expected_output"])
        else:
            print(item["text"])

        print("\nSupporting evidence:")

        for evidence_id in item["evidence_ids"]:
            print(json.dumps(
                full_evidence_lookup[evidence_id],
                indent=2,
                allow_nan=False,
            ))

print("\nHuman approval: PENDING")


EXECUTIVE_SUMMARY

Item 1:
Units sold decreased by 23,277 to 202,535 units, a 10.31% decline week-over-week, while sales revenue declined by USD 73,207.61 to USD 539,795.10, an 11.94% decrease.

Supporting evidence:
{
  "evidence_id": "overall_1",
  "metric": "Units sold",
  "previous_value": 225812.0,
  "current_value": 202535.0,
  "change": -23277.0,
  "wow_change_pct": -10.308132428746037
}
{
  "evidence_id": "overall_2",
  "metric": "Sales revenue (USD)",
  "previous_value": 613002.7100000001,
  "current_value": 539795.1,
  "change": -73207.6100000001,
  "wow_change_pct": -11.942461070033458
}

Item 2:
Store WI 2 experienced a revenue decline of USD 32,394.56, a 33.92% drop compared to the prior week.

Supporting evidence:
{
  "evidence_id": "store_WI_2",
  "store_id": "WI_2",
  "Previous revenue": 95506.05,
  "Current revenue": 63111.49,
  "Revenue change": -32394.560000000005,
  "WoW change (%)": -33.91885644940819
}

ALERT_COMMENTARY

Item 1:
Multiple alerts identified signific

In [56]:
revised_ai_draft_v2["alert_commentary"][0]["text"] = (
    "Exploratory above-expected sales alerts occurred for Foods 1 "
    "at store CA 2 on 2016-05-21 and 2016-05-22. "
    "Actual sales were 809 and 735 units, respectively, "
    "39.24% and 25.07% above expected levels."
)

revised_ai_draft_v2["proposed_actions"][0] = {
    "action": (
        "Compare daily revenue and units sold across both weeks "
        "for each department at store WI 2."
    ),
    "expected_output": (
        "Identify departments and dates with the largest revenue "
        "declines and whether units sold also decreased. "
        "Record possible explanations as hypotheses requiring "
        "additional evidence."
    ),
    "evidence_ids": ["store_WI_2"],
}

revised_ai_draft_v2["revision"]["changes"].extend([
    "Explicitly labelled the CA 2 alerts as exploratory.",
    "Made the WI 2 investigation specific and avoided promising "
    "causal explanations from sales data alone.",
])

revised_ai_file_v2.write_text(
    json.dumps(revised_ai_draft_v2, indent=2, allow_nan=False),
    encoding="utf-8",
)

validation_record_v2["remaining_numerical_and_semantic_review"] = (
    "Reviewed against supplied evidence in a guided review; "
    "no remaining discrepancies identified in the listed claims."
)
validation_record_v2["review_method"] = (
    "Guided comparison of narrative claims with cited evidence; "
    "not an automated semantic validation."
)
validation_record_v2["human_approval"] = "PENDING"

validation_file_v2.write_text(
    json.dumps(validation_record_v2, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("Revised draft and review record saved.")
print("Guided claim review: COMPLETED")
print("Human approval: PENDING")

Revised draft and review record saved.
Guided claim review: COMPLETED
Human approval: PENDING


In [57]:
from IPython.display import Markdown, display

report_text = updated_manager_file.read_text(encoding="utf-8")

required_headings = [
    "## Overall Performance",
    "## Historical Sales Alerts",
    "## Next-Week Sales Forecast",
    "## Proposed Actions",
]

for heading in required_headings:
    assert report_text.count(heading) == 1, (
        f"Expected exactly one heading: {heading}"
    )

summary_lines = ["## Executive Summary", ""]
summary_lines.extend(
    f"- {item['text']}"
    for item in revised_ai_draft_v2["executive_summary"]
)

report_text = report_text.replace(
    "## Overall Performance",
    "\n".join(summary_lines) + "\n\n## Overall Performance",
    1,
)

for heading, section in [
    ("## Historical Sales Alerts", "alert_commentary"),
    ("## Next-Week Sales Forecast", "forecast_commentary"),
]:
    commentary = "\n".join(
        f"- {item['text']}"
        for item in revised_ai_draft_v2[section]
    )
    report_text = report_text.replace(
        heading,
        heading + "\n\n" + commentary,
        1,
    )

report_text = report_text.replace(
    "## Department Performance",
    "## Department Revenue (USD)",
    1,
)

report_text = report_text.replace(
    "## Store Highlights",
    "## Store Revenue Highlights (USD)",
    1,
)

report_text = report_text.split("## Proposed Actions", 1)[0].rstrip()

action_lines = ["## Proposed Actions", ""]

for position, item in enumerate(
    revised_ai_draft_v2["proposed_actions"], start=1
):
    action_lines.extend([
        f"### {position}. Investigation",
        "",
        item["action"],
        "",
        f"**Expected output:** {item['expected_output']}",
        "",
    ])

action_lines.extend([
    "## Notes",
    "",
    (
        "Historical data; causes are unconfirmed. Historical alerts "
        "are exploratory, and forecasts are estimates. "
        "Actions are proposed, not completed."
    ),
    "",
    (
        "Model training and tuning leakage have not been audited "
        "as part of this reporting workflow."
    ),
    "",
    "*AI-assisted narrative with guided revisions; "
    "data tables generated from evidence. Human approval is pending.*",
])

combined_report = report_text + "\n\n" + "\n".join(action_lines)

combined_report_file = updated_manager_file.with_name(
    f"{evidence_for_ai['report_id']}_{run_id_v2}_manager_ai_draft.md"
)

combined_report_file.write_text(
    combined_report,
    encoding="utf-8",
)

display(Markdown(combined_report))
print(f"Combined report saved: {combined_report_file}")
print("Human approval: PENDING")

# Weekly Sales Review

**DRAFT | 2016-05-16 to 2016-05-22**

Compared with 2016-05-09 to 2016-05-15.

Scope: 10 stores across CA, TX and WI; three food departments.
All monetary values are in USD.

## Executive Summary

- Units sold decreased by 23,277 to 202,535 units, a 10.31% decline week-over-week, while sales revenue declined by USD 73,207.61 to USD 539,795.10, an 11.94% decrease.
- Store WI 2 experienced a revenue decline of USD 32,394.56, a 33.92% drop compared to the prior week.

## Overall Performance

| Metric | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Revenue | 613,002.71 | 539,795.10 | -73,207.61 | -11.94% |
| Units sold | 225,812 | 202,535 | -23,277 | -10.31% |

## Department Revenue (USD)

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 71,509.07 | 69,255.87 | -2,253.20 | -3.15% |
| Foods 2 | 168,175.53 | 130,507.89 | -37,667.64 | -22.40% |
| Foods 3 | 373,318.11 | 340,031.34 | -33,286.77 | -8.92% |

## Store Revenue Highlights (USD)

### WI 2

Revenue change: **-32,394.56 (-33.92%)**.

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 9,560.76 | 7,961.52 | -1,599.24 | -16.73% |
| Foods 2 | 37,462.51 | 20,808.44 | -16,654.07 | -44.46% |
| Foods 3 | 48,482.78 | 34,341.53 | -14,141.25 | -29.17% |

### CA 2

Revenue change: **+2,319.51 (+3.82%)**.

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 9,970.29 | 10,715.11 | +744.82 | +7.47% |
| Foods 2 | 13,926.69 | 13,946.00 | +19.31 | +0.14% |
| Foods 3 | 36,902.15 | 38,457.53 | +1,555.38 | +4.21% |

## Historical Sales Alerts

- Exploratory above-expected sales alerts occurred for Foods 1 at store CA 2 on 2016-05-21 and 2016-05-22. Actual sales were 809 and 735 units, respectively, 39.24% and 25.07% above expected levels.
- Exploratory below-expected sales alerts occurred for Foods 2 at store WI 2 from 2016-05-20 to 2016-05-22. Actual sales ranged from 815 to 850 units, 30.13% to 35.59% below expected levels.

18 daily store-department records met both thresholds: absolute deviation of at least 25% and at least 100 units.

Above expected: 15; below expected: 3.

Top 3 records by absolute unit difference:

| Date | Store | Department | Actual units | Expected units | Difference | Deviation |
| --- | --- | --- | ---: | ---: | ---: | ---: |
| 2016-05-21 | WI 2 | Foods 2 | 850 | 1,319.7 | -469.7 | -35.6% |
| 2016-05-20 | WI 2 | Foods 2 | 815 | 1,188.8 | -373.8 | -31.4% |
| 2016-05-16 | WI 1 | Foods 3 | 1,686 | 1,312.4 | +373.6 | +28.5% |

These historical alerts are exploratory. They identify records for investigation, not confirmed business issues.

## Next-Week Sales Forecast

- The forecast estimate for total units sold for 2016-05-23 to 2016-05-29 is approximately 191,131 units.

Forecast period: 2016-05-23 to 2016-05-29.

| Department | Forecast units |
| --- | ---: |
| Foods 1 | 24,744 |
| Foods 2 | 35,567 |
| Foods 3 | 130,820 |
| **Total** | **191,131** |

Model: Calendar + monthly patterns. Values are rounded for display. Forecasts are estimates, not confirmed future sales.

## Proposed Actions

### 1. Investigation

Compare daily revenue and units sold across both weeks for each department at store WI 2.

**Expected output:** Identify departments and dates with the largest revenue declines and whether units sold also decreased. Record possible explanations as hypotheses requiring additional evidence.

### 2. Investigation

Compare actual and expected daily units for Foods 1 at store CA 2 on 2016-05-21 and 2016-05-22, and examine adjacent dates in the historical review.

**Expected output:** A daily comparison showing whether above-expected sales were isolated to those dates or also appeared on adjacent dates; causes remain unconfirmed.

## Notes

Historical data; causes are unconfirmed. Historical alerts are exploratory, and forecasts are estimates. Actions are proposed, not completed.

Model training and tuning leakage have not been audited as part of this reporting workflow.

*AI-assisted narrative with guided revisions; data tables generated from evidence. Human approval is pending.*

Combined report saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T040027951984Z_manager_ai_draft.md
Human approval: PENDING


In [58]:
polished_report = combined_report_file.read_text(
    encoding="utf-8"
)

forecast_period_line = (
    f"Forecast period: {forecast_start:%Y-%m-%d} "
    f"to {forecast_end:%Y-%m-%d}."
)

polished_report = polished_report.replace(
    forecast_period_line + "\n\n",
    "",
    1,
)

polished_report = polished_report.replace(
    "### 1. Investigation",
    "### 1. Review the WI 2 revenue decline",
    1,
)

polished_report = polished_report.replace(
    "### 2. Investigation",
    "### 2. Review above-expected sales in CA 2 Foods 1",
    1,
)

polished_report_file = combined_report_file.with_name(
    combined_report_file.stem + "_polished.md"
)

polished_report_file.write_text(
    polished_report,
    encoding="utf-8",
)

display(Markdown(polished_report))
print(f"Polished draft saved: {polished_report_file}")
print("Human approval: PENDING")

# Weekly Sales Review

**DRAFT | 2016-05-16 to 2016-05-22**

Compared with 2016-05-09 to 2016-05-15.

Scope: 10 stores across CA, TX and WI; three food departments.
All monetary values are in USD.

## Executive Summary

- Units sold decreased by 23,277 to 202,535 units, a 10.31% decline week-over-week, while sales revenue declined by USD 73,207.61 to USD 539,795.10, an 11.94% decrease.
- Store WI 2 experienced a revenue decline of USD 32,394.56, a 33.92% drop compared to the prior week.

## Overall Performance

| Metric | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Revenue | 613,002.71 | 539,795.10 | -73,207.61 | -11.94% |
| Units sold | 225,812 | 202,535 | -23,277 | -10.31% |

## Department Revenue (USD)

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 71,509.07 | 69,255.87 | -2,253.20 | -3.15% |
| Foods 2 | 168,175.53 | 130,507.89 | -37,667.64 | -22.40% |
| Foods 3 | 373,318.11 | 340,031.34 | -33,286.77 | -8.92% |

## Store Revenue Highlights (USD)

### WI 2

Revenue change: **-32,394.56 (-33.92%)**.

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 9,560.76 | 7,961.52 | -1,599.24 | -16.73% |
| Foods 2 | 37,462.51 | 20,808.44 | -16,654.07 | -44.46% |
| Foods 3 | 48,482.78 | 34,341.53 | -14,141.25 | -29.17% |

### CA 2

Revenue change: **+2,319.51 (+3.82%)**.

| Department | Previous week | Current week | Change | Change (%) |
| --- | ---: | ---: | ---: | ---: |
| Foods 1 | 9,970.29 | 10,715.11 | +744.82 | +7.47% |
| Foods 2 | 13,926.69 | 13,946.00 | +19.31 | +0.14% |
| Foods 3 | 36,902.15 | 38,457.53 | +1,555.38 | +4.21% |

## Historical Sales Alerts

- Exploratory above-expected sales alerts occurred for Foods 1 at store CA 2 on 2016-05-21 and 2016-05-22. Actual sales were 809 and 735 units, respectively, 39.24% and 25.07% above expected levels.
- Exploratory below-expected sales alerts occurred for Foods 2 at store WI 2 from 2016-05-20 to 2016-05-22. Actual sales ranged from 815 to 850 units, 30.13% to 35.59% below expected levels.

18 daily store-department records met both thresholds: absolute deviation of at least 25% and at least 100 units.

Above expected: 15; below expected: 3.

Top 3 records by absolute unit difference:

| Date | Store | Department | Actual units | Expected units | Difference | Deviation |
| --- | --- | --- | ---: | ---: | ---: | ---: |
| 2016-05-21 | WI 2 | Foods 2 | 850 | 1,319.7 | -469.7 | -35.6% |
| 2016-05-20 | WI 2 | Foods 2 | 815 | 1,188.8 | -373.8 | -31.4% |
| 2016-05-16 | WI 1 | Foods 3 | 1,686 | 1,312.4 | +373.6 | +28.5% |

These historical alerts are exploratory. They identify records for investigation, not confirmed business issues.

## Next-Week Sales Forecast

- The forecast estimate for total units sold for 2016-05-23 to 2016-05-29 is approximately 191,131 units.

| Department | Forecast units |
| --- | ---: |
| Foods 1 | 24,744 |
| Foods 2 | 35,567 |
| Foods 3 | 130,820 |
| **Total** | **191,131** |

Model: Calendar + monthly patterns. Values are rounded for display. Forecasts are estimates, not confirmed future sales.

## Proposed Actions

### 1. Review the WI 2 revenue decline

Compare daily revenue and units sold across both weeks for each department at store WI 2.

**Expected output:** Identify departments and dates with the largest revenue declines and whether units sold also decreased. Record possible explanations as hypotheses requiring additional evidence.

### 2. Review above-expected sales in CA 2 Foods 1

Compare actual and expected daily units for Foods 1 at store CA 2 on 2016-05-21 and 2016-05-22, and examine adjacent dates in the historical review.

**Expected output:** A daily comparison showing whether above-expected sales were isolated to those dates or also appeared on adjacent dates; causes remain unconfirmed.

## Notes

Historical data; causes are unconfirmed. Historical alerts are exploratory, and forecasts are estimates. Actions are proposed, not completed.

Model training and tuning leakage have not been audited as part of this reporting workflow.

*AI-assisted narrative with guided revisions; data tables generated from evidence. Human approval is pending.*

Polished draft saved: D:\ai-fmcg-sales-insights\reports\drafts\weekly_sales_20160522_20261021T040027951984Z_manager_ai_draft_polished.md
Human approval: PENDING


In [1]:
from pathlib import Path
import json

project_folder = Path(r"D:\ai-fmcg-sales-insights")

evidence_path = (
    project_folder
    / "reports"
    / "evidence"
    / "weekly_sales_20160522_full_evidence_v1.json"
)

draft_path = (
    project_folder
    / "reports"
    / "drafts"
    / "weekly_sales_20160522_20261021T040027951984Z_ai_v2_raw_revised.json"
)

for path in [evidence_path, draft_path]:
    assert path.is_file(), f"File not found: {path}"

evaluation_evidence = json.loads(
    evidence_path.read_text(encoding="utf-8")
)
evaluation_draft = json.loads(
    draft_path.read_text(encoding="utf-8")
)

assert (
    evaluation_evidence["report_id"]
    == evaluation_draft["report_id"]
), "Report IDs do not match."

print("Evidence loaded: PASS")
print("Revised draft loaded: PASS")
print(f"Report ID: {evaluation_draft['report_id']}")
print("API call made: No")

Evidence loaded: PASS
Revised draft loaded: PASS
Report ID: weekly_sales_20160522
API call made: No


In [2]:
def validate_report_structure(report, evidence):
    errors = []
    evidence_ids = []

    def collect_ids(value):
        if isinstance(value, dict):
            if "evidence_id" in value:
                evidence_ids.append(value["evidence_id"])
            for child in value.values():
                collect_ids(child)
        elif isinstance(value, list):
            for child in value:
                collect_ids(child)

    collect_ids(evidence)

    if not all(isinstance(item, str) for item in evidence_ids):
        return ["Evidence package contains invalid evidence IDs."]

    if len(evidence_ids) != len(set(evidence_ids)):
        errors.append("Evidence package contains duplicate evidence IDs.")

    valid_ids = set(evidence_ids)

    if not isinstance(report, dict):
        return errors + ["Report must be a JSON object."]

    for field in ["report_id", "report_period", "comparison_period"]:
        if field not in report or report[field] != evidence.get(field):
            errors.append(f"{field}: missing or does not match evidence.")

    if report.get("status") != "draft":
        errors.append("status: must be draft.")

    section_rules = {
        "executive_summary": (2, 2, ["text"]),
        "alert_commentary": (1, 2, ["text"]),
        "forecast_commentary": (1, 1, ["text"]),
        "proposed_actions": (2, 2, ["action", "expected_output"]),
    }

    for section, (minimum, maximum, text_fields) in section_rules.items():
        items = report.get(section)

        if not isinstance(items, list):
            errors.append(f"{section}: must be a list.")
            continue

        if not minimum <= len(items) <= maximum:
            errors.append(f"{section}: incorrect item count.")

        for position, item in enumerate(items, start=1):
            label = f"{section}[{position}]"

            if not isinstance(item, dict):
                errors.append(f"{label}: must be an object.")
                continue

            for field in text_fields:
                value = item.get(field)
                if not isinstance(value, str) or not value.strip():
                    errors.append(f"{label}.{field}: missing or empty.")

            references = item.get("evidence_ids")

            if (
                not isinstance(references, list)
                or not references
                or not all(isinstance(ref, str) for ref in references)
            ):
                errors.append(f"{label}: invalid evidence_ids.")
                continue

            for reference in references:
                if reference not in valid_ids:
                    errors.append(
                        f"{label}: unknown evidence ID {reference}."
                    )

    limitations = report.get("limitations")

    if (
        not isinstance(limitations, list)
        or not limitations
        or not all(
            isinstance(item, str) and item.strip()
            for item in limitations
        )
    ):
        errors.append("limitations: must be a non-empty list of text.")

    return errors


structure_errors = validate_report_structure(
    evaluation_draft,
    evaluation_evidence,
)

print(
    "Structure validation:",
    "PASS" if not structure_errors else "FAIL",
)

for error in structure_errors:
    print(f"- {error}")

print("Numerical and semantic accuracy: Not checked by this function")

Structure validation: PASS
Numerical and semantic accuracy: Not checked by this function


In [3]:
from copy import deepcopy
import pandas as pd

test_cases = [
    {
        "name": "Valid revised draft",
        "report": deepcopy(evaluation_draft),
        "expected_error": None,
    }
]

unknown_reference = deepcopy(evaluation_draft)
unknown_reference["executive_summary"][0]["evidence_ids"] = [
    "nonexistent_evidence"
]
test_cases.append({
    "name": "Unknown evidence ID",
    "report": unknown_reference,
    "expected_error": "unknown evidence ID",
})

wrong_date = deepcopy(evaluation_draft)
wrong_date["report_period"]["end"] = "2016-05-23"
test_cases.append({
    "name": "Incorrect report period",
    "report": wrong_date,
    "expected_error": "report_period:",
})

missing_text = deepcopy(evaluation_draft)
del missing_text["executive_summary"][0]["text"]
test_cases.append({
    "name": "Missing summary text",
    "report": missing_text,
    "expected_error": ".text: missing or empty",
})

wrong_status = deepcopy(evaluation_draft)
wrong_status["status"] = "approved"
test_cases.append({
    "name": "Unexpected approved status",
    "report": wrong_status,
    "expected_error": "status: must be draft",
})

test_results = []

for case in test_cases:
    errors = validate_report_structure(
        case["report"],
        evaluation_evidence,
    )

    if case["expected_error"] is None:
        passed = not errors
    else:
        passed = any(
            case["expected_error"] in error
            for error in errors
        )

    test_results.append({
        "Test case": case["name"],
        "Expected behavior": (
            "Accept" if case["expected_error"] is None else "Reject"
        ),
        "Actual behavior": "Accept" if not errors else "Reject",
        "Test result": "PASS" if passed else "FAIL",
    })

structure_test_results = pd.DataFrame(test_results)

display(structure_test_results)

assert structure_test_results["Test result"].eq("PASS").all(), (
    "One or more validation tests failed."
)

print("All 5 structure validation tests passed.")
print("API call made: No")

,Test case,Expected behavior,Actual behavior,Test result
0,Valid revised draft,Accept,Accept,PASS
1,Unknown evidence ID,Reject,Reject,PASS
2,Incorrect report period,Reject,Reject,PASS
3,Missing summary text,Reject,Reject,PASS
4,Unexpected approved status,Reject,Reject,PASS


All 5 structure validation tests passed.
API call made: No


In [4]:
import re
import math

def validate_actual_unit_range(item, evidence):
    pattern = (
        r"actual\s+(?:sales|units)\s+"
        r"(?:ranged\s+from|between)\s+"
        r"([\d,]+(?:\.\d+)?)\s+"
        r"(?:to|and)\s+"
        r"([\d,]+(?:\.\d+)?)"
    )

    match = re.search(pattern, item["text"], flags=re.IGNORECASE)

    if match is None:
        return {
            "result": "REVIEW",
            "reason": "No supported actual-unit range phrase found.",
        }

    lookup = {
        row["evidence_id"]: row
        for row in evidence["model_evidence"]["alert_records"]
    }

    references = item["evidence_ids"]

    if not references or any(ref not in lookup for ref in references):
        return {
            "result": "REVIEW",
            "reason": "References must identify alert records.",
        }

    records = [lookup[ref] for ref in references]

    groups = {
        (row["store_id"], row["department_id"])
        for row in records
    }

    if len(groups) != 1:
        return {
            "result": "REVIEW",
            "reason": "Referenced records cover multiple store-department groups.",
        }

    reported = [
        float(value.replace(",", ""))
        for value in match.groups()
    ]

    actual_values = [row["actual_units"] for row in records]
    expected = [min(actual_values), max(actual_values)]

    matched = all(
        math.isclose(found, correct, rel_tol=0, abs_tol=0.000001)
        for found, correct in zip(reported, expected)
    )

    return {
        "result": "PASS" if matched else "FAIL",
        "reported_range": reported,
        "evidence_range": expected,
    }


original_draft_path = draft_path.with_name(
    draft_path.name.replace("_revised.json", ".json")
)

original_draft = json.loads(
    original_draft_path.read_text(encoding="utf-8")
)

original_range_result = validate_actual_unit_range(
    original_draft["alert_commentary"][1],
    evaluation_evidence,
)

revised_range_result = validate_actual_unit_range(
    evaluation_draft["alert_commentary"][1],
    evaluation_evidence,
)

print("Original AI draft:")
print(json.dumps(original_range_result, indent=2))

print("\nRevised draft:")
print(json.dumps(revised_range_result, indent=2))

assert original_range_result["result"] == "FAIL"
assert revised_range_result["result"] == "PASS"

print("\nActual-unit range regression test: PASS")
print("API call made: No")

Original AI draft:
{
  "result": "FAIL",
  "reported_range": [
    815.0,
    836.0
  ],
  "evidence_range": [
    815.0,
    850.0
  ]
}

Revised draft:
{
  "result": "PASS",
  "reported_range": [
    815.0,
    850.0
  ],
  "evidence_range": [
    815.0,
    850.0
  ]
}

Actual-unit range regression test: PASS
API call made: No


In [5]:
from copy import deepcopy

range_test_cases = []

for label, text, expected_result in [
    (
        "Correct range",
        "Actual sales ranged from 815 to 850 units.",
        "PASS",
    ),
    (
        "Incorrect lower bound",
        "Actual sales ranged from 800 to 850 units.",
        "FAIL",
    ),
    (
        "Incorrect upper bound",
        "Actual sales ranged from 815 to 836 units.",
        "FAIL",
    ),
    (
        "Reversed bounds",
        "Actual sales ranged from 850 to 815 units.",
        "FAIL",
    ),
    (
        "Unsupported wording",
        "Actual sales varied across the three dates.",
        "REVIEW",
    ),
]:
    item = deepcopy(evaluation_draft["alert_commentary"][1])
    item["text"] = text

    result = validate_actual_unit_range(
        item,
        evaluation_evidence,
    )

    range_test_cases.append({
        "Test case": label,
        "Expected result": expected_result,
        "Actual result": result["result"],
        "Test result": (
            "PASS"
            if result["result"] == expected_result
            else "FAIL"
        ),
    })

range_test_results = pd.DataFrame(range_test_cases)
display(range_test_results)

assert range_test_results["Test result"].eq("PASS").all(), (
    "One or more range validation tests failed."
)

evaluation_results = {
    "report_id": evaluation_draft["report_id"],
    "structure_tests": structure_test_results.to_dict(
        orient="records"
    ),
    "actual_unit_range_tests": range_test_results.to_dict(
        orient="records"
    ),
    "real_error_regression": {
        "original_draft": original_range_result,
        "revised_draft": revised_range_result,
    },
    "scope": [
        "Report structure, metadata and evidence ID existence.",
        "Supported actual-unit range phrases against cited alert records.",
    ],
    "limitations": [
        "These tests do not measure overall LLM accuracy.",
        "Unsupported range wording requires manual review.",
        "Other numerical and semantic claims require separate checks.",
    ],
}

evaluation_results_file = (
    project_folder
    / "reports"
    / "validation"
    / "weekly_sales_20160522_evaluation_tests_v1.json"
)

evaluation_results_file.parent.mkdir(parents=True, exist_ok=True)
evaluation_results_file.write_text(
    json.dumps(evaluation_results, indent=2, allow_nan=False),
    encoding="utf-8",
)

print("All 5 additional range tests passed.")
print(f"Evaluation results saved: {evaluation_results_file}")
print("API call made: No")

,Test case,Expected result,Actual result,Test result
0,Correct range,PASS,PASS,PASS
1,Incorrect lower bound,FAIL,FAIL,PASS
2,Incorrect upper bound,FAIL,FAIL,PASS
3,Reversed bounds,FAIL,FAIL,PASS
4,Unsupported wording,REVIEW,REVIEW,PASS


All 5 additional range tests passed.
Evaluation results saved: D:\ai-fmcg-sales-insights\reports\validation\weekly_sales_20160522_evaluation_tests_v1.json
API call made: No


In [6]:
sales_for_evaluation = pd.read_csv(
    project_folder
    / "data"
    / "processed"
    / "daily_store_dept_sales.csv"
)

sales_for_evaluation["SALES_DATE"] = pd.to_datetime(
    sales_for_evaluation["SALES_DATE"],
    errors="raise",
).dt.normalize()

evaluation_week_ends = pd.date_range(
    end=evaluation_evidence["report_period"]["end"],
    periods=4,
    freq="7D",
)

weekly_results = []

for week_end in evaluation_week_ends:
    week_start = week_end - pd.Timedelta(days=6)
    previous_start = week_start - pd.Timedelta(days=7)

    current = sales_for_evaluation.loc[
        sales_for_evaluation["SALES_DATE"].between(
            week_start, week_end
        )
    ]
    previous = sales_for_evaluation.loc[
        sales_for_evaluation["SALES_DATE"].between(
            previous_start,
            week_start - pd.Timedelta(days=1),
        )
    ]

    expected_keys = pd.MultiIndex.from_product(
        [
            pd.date_range(week_start, week_end),
            sorted(sales_for_evaluation["STORE_ID"].unique()),
            sorted(sales_for_evaluation["DEPT_ID"].unique()),
        ],
        names=["SALES_DATE", "STORE_ID", "DEPT_ID"],
    )

    for label, frame, offset in [
        ("Current week", current, 0),
        ("Previous week", previous, -7),
    ]:
        keys = pd.MultiIndex.from_frame(
            frame[["SALES_DATE", "STORE_ID", "DEPT_ID"]]
        )
        required_keys = pd.MultiIndex.from_tuples(
            [
                (date + pd.Timedelta(days=offset), store, department)
                for date, store, department in expected_keys
            ],
            names=expected_keys.names,
        )

        assert not keys.has_duplicates, f"{label}: duplicate records."
        assert required_keys.difference(keys).empty, (
            f"{label}: missing records."
        )
        assert keys.difference(required_keys).empty, (
            f"{label}: unexpected records."
        )

    current_revenue = current["TOTAL_SALES_REVENUE"].sum()
    previous_revenue = previous["TOTAL_SALES_REVENUE"].sum()

    weekly_results.append({
        "Week start": week_start.strftime("%Y-%m-%d"),
        "Week end": week_end.strftime("%Y-%m-%d"),
        "Records": len(current),
        "Revenue (USD)": float(current_revenue),
        "Units sold": float(current["TOTAL_UNITS_SOLD"].sum()),
        "Revenue WoW (%)": (
            float((current_revenue / previous_revenue - 1) * 100)
            if previous_revenue != 0
            else None
        ),
    })

multiweek_summary = pd.DataFrame(weekly_results)

display(
    multiweek_summary.style.format({
        "Revenue (USD)": "{:,.2f}",
        "Units sold": "{:,.0f}",
        "Revenue WoW (%)": "{:+.2f}%",
    }, na_rep="N/A").hide(axis="index")
)

print("Four-week record coverage: PASS")
print("API call made: No")

Week start,Week end,Records,Revenue (USD),Units sold,Revenue WoW (%)
2016-04-25,2016-05-01,210,"507,130.53","191,019",-0.29%
2016-05-02,2016-05-08,210,"575,474.88","213,122",+13.48%
2016-05-09,2016-05-15,210,"613,002.71","225,812",+6.52%
2016-05-16,2016-05-22,210,"539,795.10","202,535",-11.94%


Four-week record coverage: PASS
API call made: No


In [7]:
import numpy as np

review_for_evaluation = pd.read_csv(
    project_folder
    / "data"
    / "processed"
    / "sales_anomaly_review_monthly_patterns.csv"
)

review_for_evaluation["sales_date"] = pd.to_datetime(
    review_for_evaluation["sales_date"],
    errors="raise",
).dt.normalize()

source_units = sales_for_evaluation[
    ["SALES_DATE", "STORE_ID", "DEPT_ID", "TOTAL_UNITS_SOLD"]
].rename(columns={
    "SALES_DATE": "sales_date",
    "STORE_ID": "store_id",
    "DEPT_ID": "dept_id",
    "TOTAL_UNITS_SOLD": "source_actual_units",
})

alert_week_results = []

for week_end in evaluation_week_ends:
    week_start = week_end - pd.Timedelta(days=6)

    week_review = review_for_evaluation.loc[
        review_for_evaluation["sales_date"].between(
            week_start, week_end
        )
    ].copy()

    week_source = source_units.loc[
        source_units["sales_date"].between(week_start, week_end)
    ]

    comparison = week_review.merge(
        week_source,
        on=["sales_date", "store_id", "dept_id"],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )

    assert len(comparison) == 210, "Unexpected weekly record count."
    assert comparison["_merge"].eq("both").all(), (
        "Historical review and source records do not align."
    )

    assert np.allclose(
        comparison["actual_units"],
        comparison["source_actual_units"],
        rtol=0,
        atol=0.000001,
    ), "Actual units differ from the source."

    numeric_columns = [
        "actual_units",
        "expected_units",
        "deviation_units",
        "deviation_pct",
    ]

    assert np.isfinite(
        week_review[numeric_columns].to_numpy(dtype=float)
    ).all(), "Invalid numerical values."

    assert week_review["expected_units"].gt(0).all(), (
        "Non-positive expected units require separate handling."
    )

    difference = (
        week_review["actual_units"] - week_review["expected_units"]
    )
    deviation_pct = difference / week_review["expected_units"] * 100

    assert np.allclose(
        week_review["deviation_units"], difference,
        rtol=0, atol=0.000001,
    ), "Incorrect deviation units."

    assert np.allclose(
        week_review["deviation_pct"], deviation_pct,
        rtol=0, atol=0.000001,
    ), "Incorrect deviation percentages."

    calculated_flags = (
        deviation_pct.abs().ge(25)
        & difference.abs().ge(100)
    )

    flag_text = (
        week_review["is_alert"]
        .astype(str).str.strip().str.lower()
    )

    assert flag_text.isin(["true", "false"]).all(), (
        "Unexpected alert flag values."
    )
    assert flag_text.eq("true").eq(calculated_flags).all(), (
        "Alert flags do not match the thresholds."
    )

    calculated_types = np.select(
        [
            calculated_flags & difference.gt(0),
            calculated_flags & difference.lt(0),
        ],
        ["Above expected", "Below expected"],
        default="Within threshold",
    )

    assert week_review["alert_type"].eq(calculated_types).all(), (
        "Alert classifications do not match."
    )

    alert_week_results.append({
        "Week end": week_end.strftime("%Y-%m-%d"),
        "Reviewed records": len(week_review),
        "Alerts": int(calculated_flags.sum()),
        "Above expected": int(
            (calculated_flags & difference.gt(0)).sum()
        ),
        "Below expected": int(
            (calculated_flags & difference.lt(0)).sum()
        ),
    })

multiweek_alert_summary = pd.DataFrame(alert_week_results)

display(multiweek_alert_summary)

print("Four-week actual-unit reconciliation: PASS")
print("Four-week alert calculations and classifications: PASS")
print("Historical exploratory review; not a model accuracy test.")
print("API call made: No")

,Week end,Reviewed records,Alerts,Above expected,Below expected
0,2016-05-01,210,20,19,1
1,2016-05-08,210,38,30,8
2,2016-05-15,210,16,16,0
3,2016-05-22,210,18,15,3


Four-week actual-unit reconciliation: PASS
Four-week alert calculations and classifications: PASS
Historical exploratory review; not a model accuracy test.
API call made: No


In [8]:
multiweek_validation_summary = multiweek_summary.merge(
    multiweek_alert_summary,
    on="Week end",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

assert multiweek_validation_summary["_merge"].eq("both").all(), (
    "Weekly sales and alert summaries do not align."
)

multiweek_validation_summary = (
    multiweek_validation_summary.drop(columns="_merge")
)

validation_folder = project_folder / "reports" / "validation"
validation_folder.mkdir(parents=True, exist_ok=True)

multiweek_csv_file = (
    validation_folder / "four_week_validation_summary.csv"
)

multiweek_validation_summary.to_csv(
    multiweek_csv_file,
    index=False,
)

multiweek_validation_record = {
    "period_start": multiweek_summary["Week start"].min(),
    "period_end": multiweek_summary["Week end"].max(),
    "weeks_checked": int(len(multiweek_validation_summary)),
    "reviewed_records": int(
        multiweek_alert_summary["Reviewed records"].sum()
    ),
    "alerts": int(multiweek_alert_summary["Alerts"].sum()),
    "checks": {
        "weekly_record_coverage": "PASS",
        "actual_units_against_sales_source": "PASS",
        "deviation_calculations": "PASS",
        "alert_thresholds": "PASS",
        "alert_classifications": "PASS",
    },
    "scope": (
        "Historical sales aggregation and alert-rule validation "
        "across four weeks."
    ),
    "limitations": [
        "This is not a four-week evaluation of LLM-generated reports.",
        "Alert counts do not measure model accuracy.",
        "Historical alerts are exploratory.",
        "Model training and tuning leakage have not been audited.",
    ],
}

multiweek_json_file = (
    validation_folder / "four_week_validation_record.json"
)

multiweek_json_file.write_text(
    json.dumps(
        multiweek_validation_record,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print(f"Weeks checked: {len(multiweek_validation_summary)}")
print(
    "Reviewed records:",
    multiweek_validation_record["reviewed_records"],
)
print("Total alerts:", multiweek_validation_record["alerts"])
print(f"Summary saved: {multiweek_csv_file}")
print(f"Validation record saved: {multiweek_json_file}")
print("API call made: No")

Weeks checked: 4
Reviewed records: 840
Total alerts: 92
Summary saved: D:\ai-fmcg-sales-insights\reports\validation\four_week_validation_summary.csv
Validation record saved: D:\ai-fmcg-sales-insights\reports\validation\four_week_validation_record.json
API call made: No


In [9]:
import inspect
import importlib.util

src_folder = project_folder / "src"
src_folder.mkdir(parents=True, exist_ok=True)

validation_module_file = src_folder / "report_validation.py"

module_source = (
    '"""Validate report structure and supported actual-unit ranges."""\n\n'
    "import math\n"
    "import re\n\n"
    + inspect.getsource(validate_report_structure)
    + "\n\n"
    + inspect.getsource(validate_actual_unit_range)
)

compile(module_source, str(validation_module_file), "exec")

if validation_module_file.exists():
    existing_source = validation_module_file.read_text(
        encoding="utf-8"
    )
    if existing_source != module_source:
        raise FileExistsError(
            "report_validation.py already contains different code. "
            "Review the existing file before replacing it."
        )
else:
    validation_module_file.write_text(
        module_source,
        encoding="utf-8",
    )

spec = importlib.util.spec_from_file_location(
    "report_validation",
    validation_module_file,
)
validation_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(validation_module)

assert validation_module.validate_report_structure(
    evaluation_draft,
    evaluation_evidence,
) == []

assert validation_module.validate_actual_unit_range(
    original_draft["alert_commentary"][1],
    evaluation_evidence,
)["result"] == "FAIL"

assert validation_module.validate_actual_unit_range(
    evaluation_draft["alert_commentary"][1],
    evaluation_evidence,
)["result"] == "PASS"

print(f"Module saved: {validation_module_file}")
print("Import from saved module: PASS")
print("Structure and real-error regression checks: PASS")
print("API call made: No")

Module saved: D:\ai-fmcg-sales-insights\src\report_validation.py
Import from saved module: PASS
Structure and real-error regression checks: PASS
API call made: No


In [10]:
import subprocess
import sys
import textwrap

tests_folder = project_folder / "tests"
fixtures_folder = tests_folder / "fixtures"
fixtures_folder.mkdir(parents=True, exist_ok=True)

fixture_data = {
    "evidence": evaluation_evidence,
    "structure_cases": test_cases,
    "range_item": evaluation_draft["alert_commentary"][1],
    "original_range_item": original_draft["alert_commentary"][1],
}

fixture_file = fixtures_folder / "report_validation_cases.json"
fixture_file.write_text(
    json.dumps(fixture_data, indent=2, allow_nan=False),
    encoding="utf-8",
)

test_source = textwrap.dedent('''
    import json
    import sys
    import unittest
    from copy import deepcopy
    from pathlib import Path

    PROJECT_ROOT = Path(__file__).resolve().parents[1]
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

    from report_validation import (
        validate_report_structure,
        validate_actual_unit_range,
    )


    class ReportValidationTests(unittest.TestCase):
        @classmethod
        def setUpClass(cls):
            fixture_path = (
                Path(__file__).parent
                / "fixtures"
                / "report_validation_cases.json"
            )
            cls.data = json.loads(
                fixture_path.read_text(encoding="utf-8")
            )

        def test_structure_cases(self):
            for case in self.data["structure_cases"]:
                with self.subTest(case=case["name"]):
                    errors = validate_report_structure(
                        case["report"], self.data["evidence"]
                    )
                    if case["expected_error"] is None:
                        self.assertEqual(errors, [])
                    else:
                        self.assertTrue(any(
                            case["expected_error"] in error
                            for error in errors
                        ))

        def test_actual_unit_ranges(self):
            cases = [
                ("Actual sales ranged from 815 to 850 units.", "PASS"),
                ("Actual sales ranged from 800 to 850 units.", "FAIL"),
                ("Actual sales ranged from 815 to 836 units.", "FAIL"),
                ("Actual sales ranged from 850 to 815 units.", "FAIL"),
                ("Actual sales varied across the three dates.", "REVIEW"),
            ]
            for text, expected in cases:
                with self.subTest(text=text):
                    item = deepcopy(self.data["range_item"])
                    item["text"] = text
                    result = validate_actual_unit_range(
                        item, self.data["evidence"]
                    )
                    self.assertEqual(result["result"], expected)

        def test_real_ai_error_regression(self):
            for field, expected in [
                ("original_range_item", "FAIL"),
                ("range_item", "PASS"),
            ]:
                with self.subTest(draft=field):
                    result = validate_actual_unit_range(
                        self.data[field], self.data["evidence"]
                    )
                    self.assertEqual(result["result"], expected)


    if __name__ == "__main__":
        unittest.main()
''').lstrip()

test_file = tests_folder / "test_report_validation.py"

compile(test_source, str(test_file), "exec")
test_file.write_text(test_source, encoding="utf-8")

test_run = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-p",
        "test_report_validation.py",
        "-v",
    ],
    cwd=project_folder,
    capture_output=True,
    text=True,
)

print(test_run.stdout)
print(test_run.stderr)

assert test_run.returncode == 0, "Saved test suite failed."

print(f"Test file saved: {test_file}")
print("Independent test execution: PASS")
print("API call made: No")


test_actual_unit_ranges (test_report_validation.ReportValidationTests) ... ok
test_real_ai_error_regression (test_report_validation.ReportValidationTests) ... ok
test_structure_cases (test_report_validation.ReportValidationTests) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.012s

OK

Test file saved: D:\ai-fmcg-sales-insights\tests\test_report_validation.py
Independent test execution: PASS
API call made: No


In [11]:
from pathlib import Path
import shutil

drafts_folder = project_folder / "reports" / "drafts"

source_report = drafts_folder / (
    "weekly_sales_20160522_20261021T040027951984Z_"
    "manager_ai_draft_polished.md"
)

assert source_report.is_file(), (
    f"Report not found: {source_report}"
)

examples_folder = project_folder / "reports" / "examples"
examples_folder.mkdir(parents=True, exist_ok=True)

example_report = (
    examples_folder / "weekly_sales_review_20160522.md"
)

shutil.copy2(source_report, example_report)

assert (
    example_report.read_bytes() == source_report.read_bytes()
), "Copied report differs from the source."

print(f"Example report saved: {example_report}")
print("Copy verification: PASS")
print("Report status: DRAFT")
print("API call made: No")

Example report saved: D:\ai-fmcg-sales-insights\reports\examples\weekly_sales_review_20160522.md
Copy verification: PASS
Report status: DRAFT
API call made: No


In [14]:
evaluation_document = """
# Reporting Validation and Evaluation

## Scope

This personal portfolio project combines historical sales metrics,
exploratory model-based alerts, forecasts and AI-assisted narrative.

Tables are generated programmatically. AI-generated narrative is
reviewed against evidence and revised where needed.

## Automated Checks

The validation module checks:

- Report ID and report/comparison periods.
- Draft status, section counts and required text fields.
- Evidence ID existence and duplicate IDs in the evidence package.
- Supported actual-unit range phrases against cited alert records.

Unsupported range wording returns REVIEW for manual assessment.
A passing range check does not validate the entire statement.

## Test Results

Three unittest methods passed, covering:

- Five structure scenarios.
- Five actual-unit range scenarios.
- Two regression checks comparing the original AI error and correction.

Run this command from the project root:

    python -m unittest discover -s tests -p "test_report_validation.py" -v

These tests assess validator behavior, not overall LLM accuracy.

## Real AI Error

For WI 2 / Foods 2 on 20-22 May 2016, actual sales were
815, 850 and 836 units.

The original AI draft incorrectly reported a range of 815-836 units.
Guided evidence review corrected it to 815-850 units.

A targeted range validator was added after the error was found.
Its regression test rejects the original range and accepts the correction.

The AI also proposed using inventory and staffing data that were not
supplied. The action was revised to use available daily sales data.

## Four-Week Data Validation

Period: 25 April-22 May 2016.

- Four weeks, each containing 210 daily store-department records.
- 840 reviewed records and 92 alerts.
- Weekly alert counts: 20, 38, 16 and 18.
- Actual units reconciled to the processed sales source.
- Deviations, alert flags and classifications matched recalculated values.
- Alerts require both absolute deviation of at least 25% and
  an absolute difference of at least 100 units.

This validates data alignment and rule calculations. It does not
evaluate four weeks of LLM reports or establish model accuracy.

## Review and Approval

Guided review compared narrative figures, dates, directions of change,
units and forecast wording with cited evidence.

Original drafts and revision records are retained.
The example report remains a draft with human approval pending.

Approval status is recorded. An approval interface and automated
publication controls have not been implemented.

## Limitations

- Valid evidence IDs do not guarantee supported claims.
- Range checks support limited wording and only the referenced records.
- Complete date coverage is not verified by the range checker.
- Other numerical claims, rankings, causal statements and action
  feasibility require separate checks or manual review.
- Historical alerts are exploratory.
- Model training and tuning leakage have not been audited here.
- Forecasts estimate units sold, not revenue or guaranteed demand.
- No representative multi-report LLM benchmark has been completed.
- Business time savings and ROI have not yet been measured.

## Supporting Files

- [Example report](../reports/examples/weekly_sales_review_20160522.md)
- [Validation module](../src/report_validation.py)
- [Tests](../tests/test_report_validation.py)
- [Validator results](../reports/validation/weekly_sales_20160522_evaluation_tests_v1.json)
- [Four-week summary](../reports/validation/four_week_validation_summary.csv)
- [Four-week record](../reports/validation/four_week_validation_record.json)
""".strip()

docs_folder = project_folder / "docs"
docs_folder.mkdir(parents=True, exist_ok=True)

evaluation_doc_file = docs_folder / "evaluation.md"

evaluation_doc_file.write_text(
    evaluation_document + "\n",
    encoding="utf-8",
)

print(f"Evaluation documentation saved: {evaluation_doc_file}")
print("API call made: No")

Evaluation documentation saved: D:\ai-fmcg-sales-insights\docs\evaluation.md
API call made: No


In [15]:
required_paths = [
    "src/report_validation.py",
    "tests/test_report_validation.py",
    "tests/fixtures/report_validation_cases.json",
    "prompts/weekly_sales_review_v2.txt",
    "data/processed/daily_store_dept_sales.csv",
    "data/processed/future_sales_forecast_monthly_patterns.csv",
    "data/processed/sales_anomaly_review_monthly_patterns.csv",
    "reports/examples/weekly_sales_review_20160522.md",
    "docs/evaluation.md",
]

for relative_path in required_paths:
    path = project_folder / relative_path
    status = "FOUND" if path.is_file() else "MISSING"
    print(f"{status}: {relative_path}")

print("\nProject notebooks:")

notebook_paths = sorted(
    project_folder.rglob("*.ipynb")
)

for path in notebook_paths:
    if ".ipynb_checkpoints" not in path.parts:
        print(path.relative_to(project_folder))

print("\nCurrent Python version:")
print(sys.version.split()[0])

FOUND: src/report_validation.py
FOUND: tests/test_report_validation.py
FOUND: tests/fixtures/report_validation_cases.json
FOUND: prompts/weekly_sales_review_v2.txt
FOUND: data/processed/daily_store_dept_sales.csv
FOUND: data/processed/future_sales_forecast_monthly_patterns.csv
FOUND: data/processed/sales_anomaly_review_monthly_patterns.csv
FOUND: reports/examples/weekly_sales_review_20160522.md
FOUND: docs/evaluation.md

Project notebooks:
notebooks\01_data_inspection.ipynb
notebooks\02_sales_forecasting.ipynb
notebooks\03_ai_business_reporting.ipynb

Current Python version:
3.9.15


In [16]:
run_instructions = """
# Running the AI Reporting Extension

## Existing Example

Open the example report without an API key or database connection:

- [Weekly sales review](../reports/examples/weekly_sales_review_20160522.md)
- [Validation and evaluation](evaluation.md)

The example remains a draft pending human approval.

## Run Validation Tests

From the project root, run:

    python -m unittest discover -s tests -p "test_report_validation.py" -v

The tests use Python's standard library and bundled JSON fixtures.
They do not require pandas, Snowflake, an API key or a paid API call.

Expected result: three test methods pass.
The methods contain multiple structure, range and regression scenarios.

## Notebook Workflow

The notebooks are:

1. notebooks/01_data_inspection.ipynb
2. notebooks/02_sales_forecasting.ipynb
3. notebooks/03_ai_business_reporting.ipynb

The reporting extension reuses these processed files:

- data/processed/daily_store_dept_sales.csv
- data/processed/future_sales_forecast_monthly_patterns.csv
- data/processed/sales_anomaly_review_monthly_patterns.csv

These files are present in the development folder. Availability in a
published repository must be checked separately.

The reporting notebook calculates KPIs, prepares evidence, calls the
LLM API, supports guided review and renders a Markdown report.

The notebook currently contains development and revision cells.
A clean end-to-end rerun has not yet been verified. Do not run every
cell automatically: API generation cells make paid requests, and some
revision cells refer to particular saved drafts.

## Configuration

Set the project root in the notebook to the local checkout folder.
The development path was D:/ai-fmcg-sales-insights.

API generation requires an API key supplied through the environment
or a hidden input prompt. Do not place the key in source code,
notebook output or committed configuration files.

Viewing the saved example and running the validation tests do not
require an API key.

## Outputs

- reports/evidence/: evidence packages.
- reports/drafts/: generated drafts and revised previews.
- reports/examples/: selected demonstration report.
- reports/validation/: validation records and test summaries.

Original AI drafts are retained alongside revisions for traceability.

## Verified Environment

The demonstrated workflow ran in Python 3.9.15.
Validation tests passed in an independent Python process.
Installation and execution in a fresh environment have not yet
been verified.
""".strip()

running_doc_file = project_folder / "docs" / "running.md"
running_doc_file.write_text(
    run_instructions + "\n",
    encoding="utf-8",
)

print(f"Running instructions saved: {running_doc_file}")
print("API call made: No")

Running instructions saved: D:\ai-fmcg-sales-insights\docs\running.md
API call made: No


In [17]:
notebook_file = (
    project_folder
    / "notebooks"
    / "03_ai_business_reporting.ipynb"
)

notebook_content = json.loads(
    notebook_file.read_text(encoding="utf-8")
)

cell_inventory = []

for position, cell in enumerate(
    notebook_content["cells"], start=1
):
    source = cell.get("source", [])
    text = "".join(source) if isinstance(source, list) else source

    nonempty_lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    api_call_detected = any(
        marker in text
        for marker in [
            "client.responses.create(",
            "client.chat.completions.create(",
        ]
    )

    cell_inventory.append({
        "Cell": position,
        "Type": cell["cell_type"],
        "Lines": len(text.splitlines()),
        "API marker": api_call_detected,
        "Preview": " | ".join(nonempty_lines[:2])[:180],
    })

inventory_table = pd.DataFrame(cell_inventory)

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", 180,
):
    display(inventory_table)

print(f"Total cells: {len(cell_inventory)}")
print("Notebook inspected without executing its cells.")

,Cell,Type,Lines,API marker,Preview
0,1,code,10,False,"from pathlib import Path | project_folder = Path(r""D:\ai-fmcg-sales-insights"")"
1,2,code,6,False,"processed_folder = project_folder / ""data"" / ""processed"" | for item in sorted(processed_folder.rglob(""*"")):"
2,3,code,14,False,"import pandas as pd | sales_file = processed_folder / ""daily_store_dept_sales.csv"""
3,4,code,19,False,"sales[""SALES_DATE""] = pd.to_datetime( | sales[""SALES_DATE""], errors=""coerce"""
4,5,code,33,False,"sales[""SALES_DATE""] = pd.to_datetime( | sales[""SALES_DATE""], errors=""raise"""
5,6,code,35,False,"report_end = pd.Timestamp(""2016-05-22"") | report_start = report_end - pd.Timedelta(days=6)"
6,7,code,32,False,"def calculate_kpis(data): | units = data[""TOTAL_UNITS_SOLD""].sum()"
7,8,code,27,False,"current_store = current_week.groupby(""STORE_ID"")[ | ""TOTAL_SALES_REVENUE"""
8,9,code,27,False,"current_dept = current_week.groupby(""DEPT_ID"")[ | ""TOTAL_SALES_REVENUE"""
9,10,code,33,False,import math | checks = {


Total cells: 21
Notebook inspected without executing its cells.


In [18]:
from pathlib import Path

print("Current working folder:", Path.cwd())

Current working folder: C:\Users\17806
